In [1]:
# # Notebook Autocontenido - Ranking Naive Trinomial Glicko-2 (Baseline de Referencia)
# # y comparacion entre Rankings
#
# **Filosofia:** Reedicion metodologica del motor de referencia embebido directamente
# en el entorno de experimentación. Trazabilidad total sin imports externos.
#
# - Motor: Shelopugin-Sirotkin (SS) puro, sin calibración.
# - Probabilidad de empate: Skellam via LightGBM-Poisson (entrenamiento externo).
# - Rating dinámico: iteración secuencial pura, sin optimización.
# - Parámetros: canónicos fijos (μ₀=1500, φ₀=350, τ=0.8, ε=1e-6, draw_correction=0.0).
# - Multi-dataset: procesa lite, full y/o lite_recortado en una sola ejecución.

# ## Celda 1 - Setup, Imports y Selector Multi-Dataset

# ------------------------------------------------------------------
# Montaje de Google Drive (ejecutar solo si estás en Colab)
# ------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------------
# CONFIGURACIÓN GLOBAL
# ------------------------------------------------------------------
BASE_DATA_DIR = "/content/drive/MyDrive/TEG02-2025Arellano/Data"
# BASE_DATA_DIR = r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\AplanarDatos"

# Modo de dataset: "lite" | "full" | "lite_recortado" | "full_lite" | "full_lite_recortado"
DATASET_MODE = "full_lite"

# ------------------------------------------------------------------
# Imports estándar (notebook 100% autocontenido; sin imports de .py externos)
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
import math
from math import exp, log, sqrt, pi
from dataclasses import dataclass
import lightgbm as lgb
from scipy.special import iv
from sklearn.metrics import mean_absolute_error
import joblib
import json
import os
from scipy.stats import spearmanr
from sklearn.metrics import log_loss
from scipy.stats import spearmanr, kendalltau

# Probablemente por instalar
!pip install beautifulsoup4 lxml
!pip install datafc
!pip install tabula-py pdfplumber pypdf2 --quiet

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from typing import Optional, List, Dict, Tuple

from datafc import clubelo

import tabula
import chardet
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


# Determinismo
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------------
# Selector de datasets activos
# ------------------------------------------------------------------
DATASET_OPTIONS = {
    "lite": ["lite"],
    "full": ["full"],
    "lite_recortado": ["lite_recortado"],
    "full_lite": ["full", "lite"],
    "full_lite_recortado": ["full", "lite", "lite_recortado"],
}

if DATASET_MODE not in DATASET_OPTIONS:
    raise ValueError(
        f"DATASET_MODE inválido: {DATASET_MODE}. "
        f"Opciones válidas: {list(DATASET_OPTIONS.keys())}"
    )

ACTIVE_DATASET_NAMES = DATASET_OPTIONS[DATASET_MODE]
print(f"Modo activo: {DATASET_MODE}")
print(f"Datasets a procesar: {', '.join(ACTIVE_DATASET_NAMES)}")

print("Setup completo.")

# ===== TRAZABILIDAD - INICIO =====
import sys, platform, subprocess
from datetime import datetime
from zoneinfo import ZoneInfo

# Al correr en Colab la hora local aparece en UTC, forzamos la GMT-4
tz_caracas=ZoneInfo("America/Caracas")

print("="*70)
print("INICIO DE EJECUCION")
print("Fecha y hora UTC:", datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"))
print("Fecha y hora local:", datetime.now(tz_caracas).strftime("%Y-%m-%d %H:%M:%S"))
print("Entorno (Python):", sys.executable)
print("Version Python:", sys.version.replace("\n", " "))
print("Plataforma:", platform.platform())
print("Versiones de librerias:")
for lib in ["pandas", "numpy", "sklearn", "torch", "lightgbm", "scipy", "matplotlib", "seaborn"]:
    try:
        mod = __import__(lib)
        print(f"  {lib}: {mod.__version__}")
    except:
        print(f"  {lib}: No instalada")
print("="*70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 131.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 131.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 113.9 MB/s eta 0:00:00
Modo activo: full_lite
Datasets a procesar: full, lite
Setup completo.
INICIO DE EJECUCION
Fecha y hora UTC: 2026-09-03 18:58:30
Fecha y ho

In [2]:
# ## Celda 2 - Motor Glicko-2 Trinomial (Shelopugin-Sirotkin embebido)
#
# Reimplementación estricta del motor SS. No se importa ningún módulo externo.
# - `draw_correction = 0.0` (naive).
# - `advantage` recibido pero anulado en este baseline.
# - Paso 5 (volatilidad) con método de Illinois (regula falsi).

@dataclass
class Rating:
    mu: float = 1500.
    rd: float = 350.
    volatility: float = 0.06


class Glicko2:
    """
    Motor Glicko-2 trinomial basado estrictamente en Shelopugin-Sirotkin.
    """

    def __init__(
        self,
        mu: float = 1500.,
        rd: float = 350.,
        tau: float = 0.8,
        epsilon: float = 0.000001,
        draw_correction: float = 0.0,
    ) -> None:
        self.mu = mu
        self.rd = rd
        self.tau = tau
        self.q = log(10) / 400
        self.q1 = 400 / log(10)
        self.g_constant = 1.5 / (pi ** 2)
        self.epsilon = epsilon
        self.draw_correction = draw_correction

    # ------------------------------------------------------------------
    # Conversión de escala
    # ------------------------------------------------------------------
    def _convert_into_glicko2(self, rating: Rating) -> Rating:
        mu = (rating.mu - self.mu) * self.q
        phi = rating.rd * self.q
        return Rating(mu, phi, rating.volatility)

    def _convert_from_glicko2(self, rating: Rating) -> Rating:
        mu = rating.mu * self.q1 + self.mu
        phi = rating.rd * self.q1
        return Rating(mu, phi, rating.volatility)

    # ------------------------------------------------------------------
    # Funciones auxiliares del algoritmo
    # ------------------------------------------------------------------
    def _g(self, phi: float, phi_opp: float) -> float:
        return 1.0 / sqrt(1.0 + self.g_constant * (phi ** 2 + phi_opp ** 2))

    def _expected_score(
        self,
        mu: float,
        mu_opp: float,
        g: float,
        skellam_draw_probability: float,
    ) -> float:
        delta_mu = g * (mu - mu_opp)
        draw_coefficient = 1.0 + exp(self.draw_correction + skellam_draw_probability)
        exp_delta_mu = exp(delta_mu)

        win_probability = 1.0 / (1.0 + draw_coefficient / exp_delta_mu)
        loss_probability = 1.0 / (1.0 + draw_coefficient * exp_delta_mu)
        tie_probability = 1.0 - win_probability - loss_probability

        return win_probability + 0.5 * tie_probability

    @staticmethod
    def _v(g: float, expected_score: float) -> float:
        denom = g ** 2 * expected_score * (1.0 - expected_score)
        return 1.0 / denom if denom != 0.0 else 1e12

    @staticmethod
    def _rating_improvement(
        v: float, g: float, outcome: float, expected_score: float
    ) -> float:
        return v * g * (outcome - expected_score)

    @staticmethod
    def _f(
        rating_improvement_squared: float,
        v: float,
        phi_squared: float,
        volatility_squared: float,
        x: float,
    ) -> float:
        exp_x = exp(x)
        temp_var = phi_squared + v + exp_x
        first_term = exp_x * (rating_improvement_squared - temp_var) / (
            2.0 * temp_var ** 2
        )
        second_term = x - log(volatility_squared)
        return first_term - second_term

    # ------------------------------------------------------------------
    # Paso 5 - Actualización de volatilidad (Illinois / regula falsi)
    # ------------------------------------------------------------------
    def _update_volatility(
        self,
        rating: Rating,
        rating_opp: Rating,
        outcome: float,
        skellam_draw_probability: float,
    ) -> float:
        mu = rating.mu
        phi = rating.rd
        phi_squared = phi ** 2
        volatility_squared = rating.volatility ** 2
        mu_opp = rating_opp.mu
        phi_opp = rating_opp.rd

        g = self._g(phi, phi_opp)
        expected_score = self._expected_score(mu, mu_opp, g, skellam_draw_probability)
        v = self._v(g, expected_score)
        rating_improvement = self._rating_improvement(v, g, outcome, expected_score)
        rating_improvement_squared = rating_improvement ** 2

        a = log(volatility_squared)

        temp = rating_improvement_squared - phi_squared - v
        if temp > 0.0:
            b = log(temp)
        else:
            k = 1
            while (
                self._f(
                    rating_improvement_squared,
                    v,
                    phi_squared,
                    volatility_squared,
                    a - k * self.tau,
                )
                < 0.0
            ):
                k += 1
            b = a - k * self.tau

        f_a = self._f(
            rating_improvement_squared, v, phi_squared, volatility_squared, a
        )
        f_b = self._f(
            rating_improvement_squared, v, phi_squared, volatility_squared, b
        )

        while abs(b - a) > self.epsilon:
            c = a + (a - b) * f_a / (f_b - f_a)
            f_c = self._f(
                rating_improvement_squared, v, phi_squared, volatility_squared, c
            )
            if f_c * f_b < 0.0:
                a, f_a = b, f_b
            else:
                f_a /= 2.0
            b, f_b = c, f_c

        return exp(a / 2.0)

    # ------------------------------------------------------------------
    # Pasos 6-8 - Actualización de rating y RD
    # ------------------------------------------------------------------
    def _update_rating(
        self,
        rating: Rating,
        rating_opp: Rating,
        outcome: float,
        skellam_draw_probability: float,
    ) -> Rating:
        rating = self._convert_into_glicko2(rating)
        rating_opp = self._convert_into_glicko2(rating_opp)

        mu = rating.mu
        phi = rating.rd
        mu_opp = rating_opp.mu
        phi_opp = rating_opp.rd

        g = self._g(phi, phi_opp)
        expected_score = self._expected_score(mu, mu_opp, g, skellam_draw_probability)
        v = self._v(g, expected_score)

        new_volatility = self._update_volatility(
            rating, rating_opp, outcome, skellam_draw_probability
        )

        squared_phi_star = phi ** 2 + new_volatility ** 2
        new_phi = 1.0 / sqrt((1.0 / squared_phi_star) + (1.0 / v))
        new_mu = mu + (new_phi ** 2) * g * (outcome - expected_score)

        new_rating = Rating(new_mu, new_phi, new_volatility)
        return self._convert_from_glicko2(new_rating)

    @staticmethod
    def _increase_mu(rating: Rating, addition: float) -> Rating:
        return Rating(rating.mu + addition, rating.rd, rating.volatility)

    # ------------------------------------------------------------------
    # Interfaz pública
    # ------------------------------------------------------------------
    def rate(
        self,
        home_rating: Rating,
        away_rating: Rating,
        advantage: float,
        outcome: str,
        skellam_draw_probability: float,
    ) -> tuple[Rating, Rating]:
        """
        Actualiza ratings tras un partido.
        outcome: 'H' | 'D' | 'A'
        """
        increased_home = self._increase_mu(home_rating, advantage)
        decreased_away = self._increase_mu(away_rating, -advantage)

        if outcome == "H":
            new_home = self._update_rating(
                increased_home, decreased_away, 1.0, skellam_draw_probability
            )
            new_away = self._update_rating(
                decreased_away, increased_home, 0.0, skellam_draw_probability
            )
        elif outcome == "D":
            new_home = self._update_rating(
                increased_home, decreased_away, 0.5, skellam_draw_probability
            )
            new_away = self._update_rating(
                decreased_away, increased_home, 0.5, skellam_draw_probability
            )
        else:  # 'A'
            new_home = self._update_rating(
                increased_home, decreased_away, 0.0, skellam_draw_probability
            )
            new_away = self._update_rating(
                decreased_away, increased_home, 1.0, skellam_draw_probability
            )

        updated_home = self._increase_mu(new_home, -advantage)
        updated_away = self._increase_mu(new_away, advantage)

        return updated_home, updated_away


In [3]:
# ## Celda 3 - Bloque Skellam (Entrenamiento por Dataset Activo)
#
# Para cada dataset activo: carga, entrena dos regresores Poisson (LightGBM),
# calcula `draw_prob` via Skellam para todo el dataset, y persiste los modelos.

# ##
# ------------------------------------------------------------------
# Rolling features canónicas (32)
# ------------------------------------------------------------------
rolling_features = [
    "home_goals_r5", "home_Gcon_r5", "home_goals_r10", "home_Gcon_r10",
    "home_goals_r20", "home_Gcon_r20", "home_goals_r30", "home_Gcon_r30",
    "home_goals_home_r5", "home_Gcon_home_r5", "home_goals_home_r10", "home_Gcon_home_r10",
    "home_goals_home_r20", "home_Gcon_home_r20", "home_goals_home_r30", "home_Gcon_home_r30",
    "away_goals_r5", "away_Gcon_r5", "away_goals_r10", "away_Gcon_r10",
    "away_goals_r20", "away_Gcon_r20", "away_goals_r30", "away_Gcon_r30",
    "away_goals_away_r5", "away_Gcon_away_r5", "away_goals_away_r10", "away_Gcon_away_r10",
    "away_goals_away_r20", "away_Gcon_away_r20", "away_goals_away_r30", "away_Gcon_away_r30",
]

# ------------------------------------------------------------------
# Funciones auxiliares
# ------------------------------------------------------------------
def skellam_draw_prob(mu1, mu2):
    """Probabilidad de empate según distribución de Skellam (k=0)."""
    mu1 = np.maximum(mu1, 0.0)
    mu2 = np.maximum(mu2, 0.0)
    x = 2.0 * np.sqrt(mu1 * mu2)
    x = np.clip(x, 0, 700)
    return np.exp(-(mu1 + mu2)) * iv(0, x)


def load_base_dataframes(base_dir: str):
    """Carga los CSV base lite y full."""
    ruta_lite = os.path.join(base_dir, "matches_final_model_clean2.csv")
    ruta_full = os.path.join(base_dir, "matches_final_model_full2.csv")
    if not os.path.exists(ruta_lite):
        raise FileNotFoundError(f"No se encontró: {ruta_lite}")
    if not os.path.exists(ruta_full):
        raise FileNotFoundError(f"No se encontró: {ruta_full}")
    return pd.read_csv(ruta_lite), pd.read_csv(ruta_full)


def build_lite_recortado(df_lite_source, df_full_source):
    if (
        "match_id_new" not in df_lite_source.columns
        or "match_id_new" not in df_full_source.columns
    ):
        raise ValueError("Se requiere 'match_id_new' en ambos datasets para construir lite_recortado")
    common_ids = pd.Index(df_full_source["match_id_new"].dropna().unique())
    return df_lite_source.loc[df_lite_source["match_id_new"].isin(common_ids)].copy()


def get_available_rolling_features(df):
    cols = [c for c in rolling_features if c in df.columns]
    missing = [c for c in rolling_features if c not in df.columns]
    if missing:
        print(f"  Advertencia: faltan {len(missing)} rolling features: {missing}")
    if not cols:
        raise ValueError("No hay rolling features disponibles.")
    return cols


# ------------------------------------------------------------------
# Carga base
# ------------------------------------------------------------------
df_lite_raw, df_full_raw = load_base_dataframes(BASE_DATA_DIR)

# Construir lite_recortado si es necesario
if "lite_recortado" in ACTIVE_DATASET_NAMES:
    df_lite_recortado_raw = build_lite_recortado(df_lite_raw, df_full_raw)
    print(f"/lite_recortado/ construido: {len(df_lite_recortado_raw)} filas")
else:
    df_lite_recortado_raw = None

# Registro de datasets
DATASET_REGISTRY = {
    "lite": df_lite_raw,
    "full": df_full_raw,
    "lite_recortado": df_lite_recortado_raw,
}

# ------------------------------------------------------------------
# Entrenamiento Skellam por dataset activo
# ------------------------------------------------------------------
skellam_results = {}  # dataset_name -> {"df": df_con_draw, "model_home": ..., "model_away": ...}

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*60}")
    print(f" PROCESANDO SKELLAM: {dataset_name.upper()}")
    print(f"{'='*60}")

    df_src = DATASET_REGISTRY[dataset_name].copy()

    # Verificar columnas mínimas
    required = ["home_team", "away_team", "home_goals", "away_goals", "match_date_from_id"]
    for col in required:
        if col not in df_src.columns:
            raise ValueError(f"[{dataset_name}] Falta columna obligatoria: {col}")

    # Orden cronológico
    df_src = df_src.sort_values("match_date_from_id").reset_index(drop=True)

    # Features disponibles
    available_rolling = get_available_rolling_features(df_src)

    # Split 70/30 temporal
    n_train = int(len(df_src) * 0.7)
    train_df = df_src.iloc[:n_train].copy()
    test_df = df_src.iloc[n_train:].copy()

    print(f"  Total: {len(df_src)} | Train: {len(train_df)} | Test: {len(test_df)}")

    # Entrenar Poisson
    X_train = train_df[available_rolling]
    y_home_train = train_df["home_goals"]
    y_away_train = train_df["away_goals"]

    model_home = lgb.LGBMRegressor(
        objective="poisson", metric="mae",
        n_estimators=200, learning_rate=0.05, max_depth=5,
        random_state=42, verbose=-1,
    )
    model_away = lgb.LGBMRegressor(
        objective="poisson", metric="mae",
        n_estimators=200, learning_rate=0.05, max_depth=5,
        random_state=42, verbose=-1,
    )

    model_home.fit(X_train, y_home_train)
    model_away.fit(X_train, y_away_train)

    # Predecir sobre todo el dataset
    X_full = df_src[available_rolling]
    df_src["mu_home"] = model_home.predict(X_full)
    df_src["mu_away"] = model_away.predict(X_full)

    mu_h = np.maximum(df_src["mu_home"].values, 0.0)
    mu_a = np.maximum(df_src["mu_away"].values, 0.0)
    df_src["draw_prob"] = skellam_draw_prob(mu_h, mu_a)

    # Persistir modelos
    joblib.dump(model_home, f"skellam_naive_{dataset_name}_home_model.pkl")
    joblib.dump(model_away, f"skellam_naive_{dataset_name}_away_model.pkl")
    print(f"  Modelos guardados: skellam_naive_{dataset_name}_home/away_model.pkl")

    # Guardar en registro
    skellam_results[dataset_name] = {
        "df": df_src,
        "model_home": model_home,
        "model_away": model_away,
    }

    print(f"  draw_prob stats:")
    print(df_src["draw_prob"].describe())




 PROCESANDO SKELLAM: FULL
  Total: 26960 | Train: 18872 | Test: 8088
  Modelos guardados: skellam_naive_full_home/away_model.pkl
  draw_prob stats:
count    26960.000000
mean         0.252742
std          0.032094
min          0.037238
25%          0.237789
50%          0.255364
75%          0.274453
max          0.359825
Name: draw_prob, dtype: float64

 PROCESANDO SKELLAM: LITE
  Total: 37712 | Train: 26398 | Test: 11314
  Modelos guardados: skellam_naive_lite_home/away_model.pkl
  draw_prob stats:
count    37712.000000
mean         0.255736
std          0.032731
min          0.036609
25%          0.239552
50%          0.257894
75%          0.277005
max          0.383294
Name: draw_prob, dtype: float64


In [4]:
# ## Celda 4 - Loop Secuencial Naive (por Dataset Activo)
#
# Para cada dataset activo: instancia un motor limpio, itera cronológicamente
# partido a partido, y acumula el historial de ratings.

# Resultados globales
glicko_outputs = {}  # dataset_name -> {"ratings": dict, "history": DataFrame}

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*60}")
    print(f" EJECUTANDO GLICKO-2: {dataset_name.upper()}")
    print(f"{'='*60}")

    df = skellam_results[dataset_name]["df"].copy()
    df = df.sort_values("match_date_from_id").reset_index(drop=True)

    # Motor limpio por dataset (misma semilla conceptual)
    glicko = Glicko2(mu=1500.0, rd=350.0, tau=0.8, epsilon=0.000001, draw_correction=0.0)
    ratings = {}

    def get_rating(team: str) -> Rating:
        if team not in ratings:
            ratings[team] = Rating()
        return ratings[team]

    history = []

    for idx, row in df.iterrows():
        home_team = row["home_team"]
        away_team = row["away_team"]

        home_rating = get_rating(home_team)
        away_rating = get_rating(away_team)

        if row["home_goals"] > row["away_goals"]:
            outcome = "H"
        elif row["home_goals"] < row["away_goals"]:
            outcome = "A"
        else:
            outcome = "D"

        skellam_draw_probability = row["draw_prob"]

        new_home, new_away = glicko.rate(
            home_rating, away_rating,
            advantage=0.0,
            outcome=outcome,
            skellam_draw_probability=skellam_draw_probability,
        )

        ratings[home_team] = new_home
        ratings[away_team] = new_away

        history.append({
            "match_id": row.get("match_id_new", idx),
            "match_date": row["match_date_from_id"],
            "home_team": home_team,
            "away_team": away_team,
            "home_mu": new_home.mu,
            "home_rd": new_home.rd,
            "home_volatility": new_home.volatility,
            "away_mu": new_away.mu,
            "away_rd": new_away.rd,
            "away_volatility": new_away.volatility,
            "outcome": outcome,
            "draw_prob": skellam_draw_probability,
        })

    df_history = pd.DataFrame(history)
    glicko_outputs[dataset_name] = {
        "ratings": ratings,
        "history": df_history,
    }

    print(f"  Partidos procesados : {len(df_history)}")
    print(f"  Equipos únicos      : {len(ratings)}")


 EJECUTANDO GLICKO-2: FULL


  Partidos procesados : 26960
  Equipos únicos      : 336

 EJECUTANDO GLICKO-2: LITE
  Partidos procesados : 37712
  Equipos únicos      : 414


In [5]:
# ## Celda 5 - Exportación y Resumen (por Dataset Activo)

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*70}")
    print(f" EXPORTACIÓN Y RESUMEN: {dataset_name.upper()}")
    print(f"{'='*70}")

    ratings = glicko_outputs[dataset_name]["ratings"]
    df_history = glicko_outputs[dataset_name]["history"]

    # Conteo de partidos
    home_counts = df_history["home_team"].value_counts()
    away_counts = df_history["away_team"].value_counts()
    n_partidos = home_counts.add(away_counts, fill_value=0).astype(int)

    # Tabla final
    records = []
    for team, rating in ratings.items():
        records.append({
            "team": team,
            "mu": rating.mu,
            "rd": rating.rd,
            "volatility": rating.volatility,
            "n_partidos": n_partidos.get(team, 0),
        })
    df_ratings_final = pd.DataFrame(records)

    # Exportar
    out_final = os.path.join(BASE_DATA_DIR, f"ratings_final_naive_trinomial_{dataset_name}.csv")
    out_history = os.path.join(BASE_DATA_DIR, f"ratings_history_naive_trinomial_{dataset_name}.csv")

    df_ratings_final.to_csv(out_final, index=False)
    df_history.to_csv(out_history, index=False)

    print(f"  CSV exportado: {out_final}")
    print(f"  CSV exportado: {out_history}")

    # Resumen
    top20 = df_ratings_final.sort_values("mu", ascending=False).head(20)
    print("\n  >>> Top 20 equipos por rating mu descendente:")
    print(top20[["team", "mu", "rd", "volatility", "n_partidos"]].to_string(index=False))

    mu_range = df_ratings_final["mu"].max() - df_ratings_final["mu"].min()
    print(f"\n  >>> Rango de ratings (max(mu) − min(mu)): {mu_range:.2f}")
    print(f"  >>> Número total de equipos unicos      : {df_ratings_final['team'].nunique()}")

    corr, pval = spearmanr(df_ratings_final["mu"], df_ratings_final["n_partidos"])
    print(f"  >>> Correlación Spearman (mu vs n_partidos): {corr:.4f} (p = {pval:.4f})")

    print(f"\n  >>> Estadísticas descriptivas de draw_prob:")
    print(skellam_results[dataset_name]["df"]["draw_prob"].describe())



 EXPORTACIÓN Y RESUMEN: FULL
  CSV exportado: /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_final_naive_trinomial_full.csv
  CSV exportado: /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_history_naive_trinomial_full.csv

  >>> Top 20 equipos por rating mu descendente:
      team          mu        rd  volatility  n_partidos
       PSG 1833.971468 64.078015    0.059504         270
RealMadrid 1795.323728 64.612293    0.059334         289
   AlHilal 1779.653205 68.013447    0.059513         197
    Bayern 1777.745753 64.831162    0.059679         250
 Liverpool 1772.714447 65.371109    0.059578         266
Olympiacos 1765.589180 63.442262    0.059400         242
InterMilan 1756.126151 64.136947    0.059556         177
 Barcelona 1747.436279 64.118198    0.059384         264
   ManCity 1743.448952 64.526443    0.059560         276
   Arsenal 1740.492345 63.983533    0.059306         252
 AtlMadrid 1728.926191 63.721203    0.059422         264
 AlIttihad 1714.776090 64.360

In [6]:
# ## Celda 6 - Sanity Check (por Dataset Activo)

print("=" * 70)
print(" SANITY CHECK MULTI-DATASET")
print("=" * 70)


def decompose_probabilities(home_mu, home_rd, away_mu, away_rd, draw_prob):
    """Reconstruye la tríada (W, T, L) usando la misma lógica del motor SS."""
    q = log(10) / 400
    g_c = 1.5 / (pi ** 2)

    mu = (home_mu - 1500.0) * q
    phi = home_rd * q
    mu_opp = (away_mu - 1500.0) * q
    phi_opp = away_rd * q

    g = 1.0 / sqrt(1.0 + g_c * (phi ** 2 + phi_opp ** 2))
    delta_mu = g * (mu - mu_opp)

    draw_coefficient = 1.0 + exp(0.0 + draw_prob)
    exp_delta_mu = exp(delta_mu)

    win = 1.0 / (1.0 + draw_coefficient / exp_delta_mu)
    loss = 1.0 / (1.0 + draw_coefficient * exp_delta_mu)
    tie = 1.0 - win - loss
    return win, tie, loss


for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n--- Dataset: {dataset_name} ---")

    out_final = os.path.join(BASE_DATA_DIR, f"ratings_final_naive_trinomial_{dataset_name}.csv")
    out_history = os.path.join(BASE_DATA_DIR, f"ratings_history_naive_trinomial_{dataset_name}.csv")

    # 1. Existencia CSVs
    for fname in [out_final, out_history]:
        exists = os.path.exists(fname)
        size = os.path.getsize(fname) if exists else 0
        status = "OK" if exists and size > 0 else "FALLO"
        print(f"  [{status}] {fname}")

    df_ratings_final = pd.read_csv(out_final)
    df_history = pd.read_csv(out_history)

    # 2. NaN en finales
    nan_final = df_ratings_final[["team", "mu", "rd", "volatility"]].isna().sum().sum()
    print(f"  [NaN finales: {nan_final}] {'OK' if nan_final == 0 else 'FALLO'}")

    # 3. NaN en historial
    nan_hist = df_history[[
        "match_id", "match_date", "home_team", "away_team",
        "home_mu", "home_rd", "home_volatility",
        "away_mu", "away_rd", "away_volatility",
        "outcome", "draw_prob",
    ]].isna().sum().sum()
    print(f"  [NaN historial: {nan_hist}] {'OK' if nan_hist == 0 else 'FALLO'}")

    # 4. Suma de probabilidades en muestra aleatoria
    np.random.seed(42)
    sample = df_history.sample(n=min(5, len(df_history))).reset_index(drop=True)

    print(f"  Suma de probabilidades (muestra n=5):")
    for i, row in sample.iterrows():
        w, t, l = decompose_probabilities(
            row["home_mu"], row["home_rd"],
            row["away_mu"], row["away_rd"],
            row["draw_prob"],
        )
        total = w + t + l
        ok = abs(total - 1.0) < 1e-6
        print(f"    {i+1}. {row['home_team']} vs {row['away_team']} | "
              f"Sum={total:.10f} | {'OK' if ok else 'DESVIO'}")

print("\n[SANITY CHECK COMPLETADO PARA TODOS LOS DATASETS ACTIVOS]")

 SANITY CHECK MULTI-DATASET

--- Dataset: full ---
  [OK] /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_final_naive_trinomial_full.csv
  [OK] /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_history_naive_trinomial_full.csv
  [NaN finales: 0] OK
  [NaN historial: 0] OK
  Suma de probabilidades (muestra n=5):
    1. AlHilal vs AlEttifaq | Sum=1.0000000000 | OK
    2. Freiburg vs Hertha | Sum=1.0000000000 | OK
    3. AlRaed vs Abha | Sum=1.0000000000 | OK
    4. Brighton vs SheffieldUtd | Sum=1.0000000000 | OK
    5. NERevolution vs DCUtd | Sum=1.0000000000 | OK

--- Dataset: lite ---
  [OK] /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_final_naive_trinomial_lite.csv
  [OK] /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_history_naive_trinomial_lite.csv
  [NaN finales: 0] OK
  [NaN historial: 0] OK
  Suma de probabilidades (muestra n=5):
    1. Austin vs AtlantaUtd | Sum=1.0000000000 | OK
    2. Damac vs AlTaawoun | Sum=1.0000000000 | OK
    3. Celta vs Atl

In [7]:
# ## Celda 7 - POST-PROCESAMIENTO: MÉTRICAS COMPLETAS DEL NAIVE (corregida)

# Función vectorizada para descomponer probabilidades (usando np en lugar de math)
def decompose_probabilities(home_mu, home_rd, away_mu, away_rd, draw_prob):
    q = np.log(10) / 400
    g_c = 1.5 / (np.pi ** 2)
    # Conversión a escala Glicko-2
    mu = (home_mu - 1500.0) * q
    phi = home_rd * q
    mu_opp = (away_mu - 1500.0) * q
    phi_opp = away_rd * q
    g = 1.0 / np.sqrt(1.0 + g_c * (phi ** 2 + phi_opp ** 2))
    delta_mu = g * (mu - mu_opp)
    draw_coefficient = 1.0 + np.exp(0.0 + draw_prob)
    exp_delta_mu = np.exp(delta_mu)
    win = 1.0 / (1.0 + draw_coefficient / exp_delta_mu)
    loss = 1.0 / (1.0 + draw_coefficient * exp_delta_mu)
    tie = 1.0 - win - loss
    return win, tie, loss

# Mapeo de resultados a índices para multiclase
outcome_to_idx = {'H': 0, 'D': 1, 'A': 2}

print("=" * 70)
print(" CÁLCULO DE MÉTRICAS COMPLETAS PARA EL MOTOR NAIVE (basado en CSVs generados)")
print("=" * 70)

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*60}")
    print(f" DATASET: {dataset_name.upper()}")
    print(f"{'='*60}")
    
    # 1. Cargar historial (contiene las probabilidades base y los outcomes)
    hist_path = os.path.join(BASE_DATA_DIR, f"ratings_history_naive_trinomial_{dataset_name}.csv")
    df_hist = pd.read_csv(hist_path)
    
    # 2. Reconstruir el vector de probabilidades (P_H, P_D, P_A) para cada partido
    P_H, P_D, P_A = decompose_probabilities(
        df_hist["home_mu"].values,
        df_hist["home_rd"].values,
        df_hist["away_mu"].values,
        df_hist["away_rd"].values,
        df_hist["draw_prob"].values
    )
    
    # 3. Recortar para evitar log(0) en la log-loss
    eps = 1e-15
    P_H = np.clip(P_H, eps, 1 - eps)
    P_D = np.clip(P_D, eps, 1 - eps)
    P_A = np.clip(P_A, eps, 1 - eps)
    probas = np.column_stack([P_H, P_D, P_A])  # shape (N, 3)
    
    # Verificar que las probabilidades suman 1 (dentro de tolerancia)
    sum_probas = np.sum(probas, axis=1)
    if not np.allclose(sum_probas, 1.0, atol=1e-6):
        print("  Advertencia: algunas filas no suman 1. Re-normalizando...")
        probas = probas / sum_probas[:, np.newaxis]
        # Reasignar P_H, P_D, P_A por si se usan más adelante
        P_H, P_D, P_A = probas[:, 0], probas[:, 1], probas[:, 2]
    
    # 4. Target multiclase: índices 0,1,2
    y_true_indices = df_hist["outcome"].map(outcome_to_idx).values
    
    # 5. Log-Loss CATEGÓRICA (H/D/A) con y_true como índices
    ll_cat = log_loss(y_true_indices, probas)
    
    # 6. Log-Loss BINARIA (H vs no H)
    y_true_bin = (df_hist["outcome"] == "H").astype(int)
    ll_bin = log_loss(y_true_bin, P_H)
    
    # 7. Cargar ratings finales para Spearman y rango
    final_path =  os.path.join(BASE_DATA_DIR, f"ratings_final_naive_trinomial_{dataset_name}.csv")
    df_final = pd.read_csv(final_path)
    
    # Contar partidos por equipo (si no está en el CSV final)
    home_counts = df_hist["home_team"].value_counts()
    away_counts = df_hist["away_team"].value_counts()
    n_partidos = home_counts.add(away_counts, fill_value=0).astype(int)
    df_final["n_partidos"] = df_final["team"].map(n_partidos).fillna(0).astype(int)
    
    rango = df_final["mu"].max() - df_final["mu"].min()
    n_equipos = df_final["team"].nunique()
    corr, pval = spearmanr(df_final["mu"], df_final["n_partidos"])
    
    # 8. Imprimir resultados
    print(f"  >>> Log-Loss CATEGÓRICA (H/D/A) : {ll_cat:.6f}")
    print(f"  >>> Log-Loss BINARIA (H vs no H): {ll_bin:.6f}")
    print(f"  >>> Rango de ratings (max-min)   : {rango:.2f}")
    print(f"  >>> Equipos únicos               : {n_equipos}")
    print(f"  >>> Spearman (mu vs n_partidos)   : {corr:.4f} (p = {pval:.4f})")

 CÁLCULO DE MÉTRICAS COMPLETAS PARA EL MOTOR NAIVE (basado en CSVs generados)

 DATASET: FULL
  >>> Log-Loss CATEGÓRICA (H/D/A) : 0.984692
  >>> Log-Loss BINARIA (H vs no H): 0.620071
  >>> Rango de ratings (max-min)   : 614.55
  >>> Equipos únicos               : 336
  >>> Spearman (mu vs n_partidos)   : 0.5976 (p = 0.0000)

 DATASET: LITE
  >>> Log-Loss CATEGÓRICA (H/D/A) : 0.985102
  >>> Log-Loss BINARIA (H vs no H): 0.620036
  >>> Rango de ratings (max-min)   : 731.12
  >>> Equipos únicos               : 414
  >>> Spearman (mu vs n_partidos)   : 0.5775 (p = 0.0000)


In [8]:
# ## Celda 8 - COMPARACIÓN NAIVE vs G2Av1513 (por dataset)
# =============================================================================
# Compara los ratings finales y rankings entre el motor Naive (baseline)
# y el motor calibrado G2Av1513 (cuerpo del trabajo).
# =============================================================================

# Directorio donde están los CSVs (el mismo donde se guardaron los Naive)
COMPARE_DIR = BASE_DATA_DIR  

# Prefijos de archivos
NAIVE_PREFIX = "ratings_final_naive_trinomial"
G2_PREFIX = "glicko_ratings_final"

# Sufijos según dataset
DATASET_SUFFIX = {
    "full": "full",
    "lite": "lite",
    "lite_recortado": "lite_recortado",
}

# Diccionario para almacenar resultados de comparación
comparison_results = {}

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*70}")
    print(f" COMPARACIÓN NAIVE vs G2Av1513 - {dataset_name.upper()}")
    print(f"{'='*70}")
    
    suffix = DATASET_SUFFIX[dataset_name]
    
    # 1. Construir rutas
    naive_file = os.path.join(COMPARE_DIR, f"{NAIVE_PREFIX}_{suffix}.csv")
    g2_file = os.path.join(COMPARE_DIR, f"{G2_PREFIX}_{suffix}.csv")
    
    # 2. Verificar existencia
    if not os.path.exists(naive_file):
        print(f"  ERROR: No se encuentra {naive_file}")
        continue
    if not os.path.exists(g2_file):
        print(f"  ERROR: No se encuentra {g2_file}")
        continue
    
    # 3. Cargar CSVs
    df_naive = pd.read_csv(naive_file)
    df_g2 = pd.read_csv(g2_file)
    
    # 4. Normalizar nombre de columna de equipo (si es necesario)
    # Asumimos que la columna se llama 'team'. Si se llama 'home_team' o similar, renombrar.
    if 'team' not in df_naive.columns:
        # Buscar columna que contenga 'team' o 'Team'
        possible = [c for c in df_naive.columns if 'team' in c.lower()]
        if possible:
            df_naive.rename(columns={possible[0]: 'team'}, inplace=True)
        else:
            print("  ERROR: No se encontro columna de equipo en df_naive")
            continue
    if 'team' not in df_g2.columns:
        possible = [c for c in df_g2.columns if 'team' in c.lower()]
        if possible:
            df_g2.rename(columns={possible[0]: 'team'}, inplace=True)
        else:
            print("  ERROR: No se encontro columna de equipo en df_g2")
            continue
    
    # 5. Unir por equipo (inner join para solo equipos en ambos)
    df_merged = pd.merge(df_naive, df_g2, on='team', suffixes=('_naive', '_g2'))
    
    # 6. Calcular rankings (orden descendente por mu)
    df_naive_sorted = df_naive.sort_values('mu', ascending=False).reset_index(drop=True)
    df_g2_sorted = df_g2.sort_values('mu', ascending=False).reset_index(drop=True)
    df_naive_sorted['rank_naive'] = df_naive_sorted.index + 1
    df_g2_sorted['rank_g2'] = df_g2_sorted.index + 1
    
    # Unir rankings al merged
    df_merged = pd.merge(df_merged, df_naive_sorted[['team', 'rank_naive']], on='team', how='left')
    df_merged = pd.merge(df_merged, df_g2_sorted[['team', 'rank_g2']], on='team', how='left')
    
    # 7. Calcular diferencias
    df_merged['mu_diff'] = df_merged['mu_naive'] - df_merged['mu_g2']
    df_merged['mu_diff_abs'] = np.abs(df_merged['mu_diff'])
    df_merged['rank_diff'] = df_merged['rank_naive'] - df_merged['rank_g2']
    df_merged['rank_diff_abs'] = np.abs(df_merged['rank_diff'])
    
    # 8. Estadísticas globales
    n_equipos = len(df_merged)
    avg_mu_diff = df_merged['mu_diff_abs'].mean()
    std_mu_diff = df_merged['mu_diff_abs'].std()
    max_mu_diff = df_merged['mu_diff_abs'].max()
    avg_rank_diff = df_merged['rank_diff_abs'].mean()
    std_rank_diff = df_merged['rank_diff_abs'].std()
    max_rank_diff = df_merged['rank_diff_abs'].max()
    
    # Correlación de Spearman entre los ratings (no rankings)
    corr, pval = spearmanr(df_merged['mu_naive'], df_merged['mu_g2'])
    
    print(f"\n  >>> Equipos en comun: {n_equipos}")
    print(f"  >>> Diferencia absoluta de mu: promedio={avg_mu_diff:.2f}, std={std_mu_diff:.2f}, max={max_mu_diff:.2f}")
    print(f"  >>> Diferencia absoluta de ranking: promedio={avg_rank_diff:.2f}, std={std_rank_diff:.2f}, max={max_rank_diff:.2f}")
    print(f"  >>> Correlación Spearman (mu_naive vs mu_g2): {corr:.4f} (p={pval:.4f})")
    
    # 9. Top 5 equipos con mayor diferencia de rating
    top5_mu = df_merged.nlargest(5, 'mu_diff_abs')[
        ['team', 'mu_naive', 'mu_g2', 'mu_diff_abs', 'rank_naive', 'rank_g2']
    ]
    print("\n  >>> Top 5 equipos con mayor diferencia absoluta de mu:")
    print(top5_mu.to_string(index=False))
    
    # 10. Top 5 equipos con mayor diferencia de ranking
    top5_rank = df_merged.nlargest(5, 'rank_diff_abs')[
        ['team', 'rank_naive', 'rank_g2', 'rank_diff_abs', 'mu_naive', 'mu_g2']
    ]
    print("\n  >>> Top 5 equipos con mayor diferencia absoluta de ranking:")
    print(top5_rank.to_string(index=False))
    
    # Guardar resultados
    comparison_results[dataset_name] = {
        'df_merged': df_merged,
        'n_equipos': n_equipos,
        'avg_mu_diff': avg_mu_diff,
        'std_mu_diff': std_mu_diff,
        'max_mu_diff': max_mu_diff,
        'avg_rank_diff': avg_rank_diff,
        'std_rank_diff': std_rank_diff,
        'max_rank_diff': max_rank_diff,
        'spearman_corr': corr,
        'spearman_p': pval,
        'top5_mu': top5_mu,
        'top5_rank': top5_rank,
    }

# 11. Resumen global
print("\n" + "="*70)
print(" RESUMEN COMPARATIVO GLOBAL (todos los datasets activos)")
print("="*70)
for ds, res in comparison_results.items():
    print(f"\n{ds.upper()}:")
    print(f"  Avg |mu_naive - mu_g2|: {res['avg_mu_diff']:.2f}  (std={res['std_mu_diff']:.2f})")
    print(f"  Avg |rank_naive - rank_g2|: {res['avg_rank_diff']:.2f}  (std={res['std_rank_diff']:.2f})")
    print(f"  Spearman: {res['spearman_corr']:.4f} (p={res['spearman_p']:.4f})")
    print(f"  Equipos en comun: {res['n_equipos']}")

# Usamos Lite porque tiene más equipos.
df_raw = pd.read_csv(os.path.join(BASE_DATA_DIR, "matches_final_model_clean2.csv"))

# Construir mapeo: 'team' -> 'Competition'

EXCLUDED_COMPETITIONS = ['CONCACAFChampionsCup','CONCACAFChampionsLeague', 
'ChampionsLeague', 'CopaLibertadores', 'CopaSudamericana', 
'EuropaConferenceLeague', 'EuropaLeague', 'FIFAClubWorldCup', 
'AFCChampionsLeagueElite','SuperligaArgentina',
]
df_filter = df_raw[~df_raw['Competition'].isin(EXCLUDED_COMPETITIONS)]
team_league_map = pd.concat([
    df_filter[['home_team', 'Competition']].rename(columns={'home_team': 'team'}),
    df_filter[['away_team', 'Competition']].rename(columns={'away_team': 'team'})
]).drop_duplicates(subset=['team']).set_index('team')['Competition'].to_dict()

# Diccionarios para almacenar los resultados
top10_results = {}
top_league_results = {}

for dataset_name, res in comparison_results.items():
    print(f"\n{'='*70}")
    print(f" TOP 10 Y TOP POR LIGA - {dataset_name.upper()}")
    print(f"{'='*70}")
    
    df_merged = res['df_merged'].copy()
    df_merged['league'] = df_merged['team'].map(team_league_map)
    
    n_mapped = df_merged['league'].notna().sum()
    n_total = len(df_merged)
    print(f"  Equipos mapeados a liga: {n_mapped}/{n_total}")
    
    # --- 1. TOP 10 GLOBAL ---
    top10 = df_merged.nlargest(10, 'mu_diff_abs')[
        ['team', 'league', 'mu_naive', 'mu_g2', 'mu_diff_abs', 'rank_naive', 'rank_g2', 'rank_diff_abs']
    ].reset_index(drop=True)
    top10_results[dataset_name] = top10
    
    print("\n  >>> TOP 10 GLOBAL (mayor |Δμ|):")
    print(top10.to_string(index=False))
    
    # --- 2. TOP POR LIGA ---
    top_league = (
        df_merged.loc[df_merged.groupby('league')['mu_diff_abs'].idxmax()]
        .sort_values('mu_diff_abs', ascending=False)
        .head(10)
        [['league', 'team', 'mu_naive', 'mu_g2', 'mu_diff_abs', 'rank_naive', 'rank_g2']]
        .reset_index(drop=True)
    )
    top_league_results[dataset_name] = top_league
    
    print("\n  >>> TOP 10 LIGAS (mayor |Δμ| del equipo más discrepante de cada liga):")
    print(top_league.to_string(index=False))

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*70}")
    print(f" TOP 10 - {dataset_name.upper()}")
    print(f"{'='*70}")
    
    suffix = {
        "full": "full",
        "lite": "lite",
        "lite_recortado": "lite_recortado"
    }[dataset_name]
    
    # 1. Cargar CSVs
    naive_file = os.path.join(BASE_DATA_DIR, f"{NAIVE_PREFIX}_{suffix}.csv")
    g2_file = os.path.join(BASE_DATA_DIR, f"{G2_PREFIX}_{suffix}.csv")
    
    df_naive = pd.read_csv(naive_file)
    df_g2 = pd.read_csv(g2_file)
    
    # 2. Ordenar y tomar Top 10
    top_naive = df_naive.nlargest(10, 'mu')[['team', 'mu', 'rd', 'volatility']].reset_index(drop=True)
    top_g2 = df_g2.nlargest(10, 'mu')[['team', 'mu', 'rd', 'volatility']].reset_index(drop=True)
    
    # 3. Imprimir
    print("\n  >>> TOP 10 NAIVE (mayor mu):")
    print(top_naive.to_string(index=False))
    
    print("\n  >>> TOP 10 G2 (mayor mu):")
    print(top_g2.to_string(index=False))


 COMPARACIÓN NAIVE vs G2Av1513 - FULL



  >>> Equipos en comun: 336
  >>> Diferencia absoluta de mu: promedio=40.91, std=27.31, max=233.22
  >>> Diferencia absoluta de ranking: promedio=23.24, std=23.65, max=201.00
  >>> Correlación Spearman (mu_naive vs mu_g2): 0.9416 (p=0.0000)

  >>> Top 5 equipos con mayor diferencia absoluta de mu:
         team    mu_naive       mu_g2  mu_diff_abs  rank_naive  rank_g2
          CSA 1232.969284 1466.184865   233.215580         334      133
     Veracruz 1219.421730 1372.775367   153.353637         336      240
      Toronto 1385.835463 1270.220809   115.614654         264      327
  Southampton 1273.666395 1172.034415   101.631980         328      336
SJEarthquakes 1419.100387 1320.344479    98.755908         220      297

  >>> Top 5 equipos con mayor diferencia absoluta de ranking:
       team  rank_naive  rank_g2  rank_diff_abs    mu_naive       mu_g2
        CSA         334      133            201 1232.969284 1466.184865
     Xanthi         295      177            118 1356.745082 1

In [9]:
# ## Celda 9 - Definiciones para Webscrap y parseo
import unicodedata
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
TIMEOUT = 15

def fetch_html(url: str) -> Optional[str]:
    try:
        response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        response.raise_for_status()
        if response.encoding is None:
            response.encoding = 'utf-8'
        return response.text
    except Exception as e:
        print(f"Error al obtener {url}: {e}")
        return None

def parse_uefa(html: str) -> pd.DataFrame:
    """
    Extrae la tabla de clubes del ranking UEFA usando pd.read_html.
    Esto es más robusto porque detecta automáticamente los encabezados.
    """
    # pd.read_html devuelve una lista de DataFrames
    tables = pd.read_html(html)
    
    # Buscar la tabla que tenga al menos 10 filas y que contenga 'TotalPoints' o 'Points'
    df = None
    for table in tables:
        # Verificar si tiene al menos 10 filas
        if len(table) < 10:
            continue
        # Verificar si tiene columnas con nombres similares a puntos o ranking
        cols = table.columns.tolist()
        if any('total' in str(c).lower() or 'point' in str(c).lower() for c in cols):
            df = table
            break
    
    if df is None:
        # Si no se encuentra, tomar la tabla más grande
        tables = [t for t in tables if len(t) > 10]
        if tables:
            df = max(tables, key=len)
        else:
            raise ValueError("No se encontró una tabla válida en UEFA")
    
    # --- LIMPIEZA DE NOMBRES DE COLUMNAS ---
    # Las primeras columnas suelen ser: Rank, (vacía), Team, Country
    # Pero pd.read_html ya asigna nombres como 'Rank', 'Unnamed: 1', 'Team', 'Country', etc.
    # Vamos a renombrar explícitamente las primeras 4 columnas si son genéricas
    new_names = {}
    for i, col in enumerate(df.columns):
        col_str = str(col).strip()
        if i == 0:
            # Si la primera columna se llama 'Rank' o similar, ya está bien
            if 'rank' not in col_str.lower():
                new_names[col] = 'rank'
        elif i == 1:
            # Segunda columna suele estar vacía o ser 'Unnamed'
            new_names[col] = 'drop_me'
        elif i == 2:
            # Tercera columna suele ser el equipo
            if 'team' not in col_str.lower() and 'club' not in col_str.lower():
                new_names[col] = 'team'
        elif i == 3:
            # Cuarta columna suele ser el país
            if 'country' not in col_str.lower() and 'nation' not in col_str.lower():
                new_names[col] = 'country'
        elif 'total' in col_str.lower() or 'point' in col_str.lower():
            new_names[col] = 'points'
        elif '/' in col_str:
            # Columnas de temporadas (las mantenemos pero no son necesarias)
            new_names[col] = col_str.replace('/', '_')
        else:
            # Cualquier otra columna la estandarizamos
            new_names[col] = standardize_colname(col_str)
    
    df.rename(columns=new_names, inplace=True)
    
    # Eliminar la columna vacía (drop_me) si existe
    if 'drop_me' in df.columns:
        df.drop(columns=['drop_me'], inplace=True)
    
    # --- CONVERTIR A NUMÉRICO ---
    # Rank
    if 'rank' in df.columns:
        df['rank'] = pd.to_numeric(df['rank'], errors='coerce')
    
    # Points (buscar la columna de puntos totales)
    points_col = None
    for col in df.columns:
        if 'points' in col.lower() or 'total' in col.lower():
            points_col = col
            break
    
    if points_col is None:
        # Buscar la última columna numérica que no sea rank
        for col in df.columns:
            if col != 'rank':
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if df[col].notna().sum() > 0:
                        points_col = col
                        break
                except:
                    pass
    
    if points_col:
        df['points'] = pd.to_numeric(df[points_col], errors='coerce')
        if points_col != 'points':
            df.drop(columns=[points_col], inplace=True)
    
    # --- FILTRAR FILAS VÁLIDAS ---
    # Eliminar filas sin rank o sin points
    df = df.dropna(subset=['rank', 'points'])
    
    # Eliminar filas donde rank no sea numérico o sea 0
    df = df[df['rank'] > 0]
    
    # --- LIMPIEZA DE PAÍSES (opcional, para mantener compatibilidad) ---
    if 'country' in df.columns:
        df['country'] = df['country'].astype(str).str.strip()
    
    # --- SELECCIONAR COLUMNAS FINALES ---
    cols_to_keep = ['rank', 'team', 'country', 'points']
    # Si alguna columna no existe, omitirla
    df = df[[c for c in cols_to_keep if c in df.columns]]
    
    # Ordenar por rank
    df = df.sort_values('rank').reset_index(drop=True)
    
    return df

def load_elo_txt(file_path: str) -> pd.DataFrame:
    # Carga archivos ELO en formato TXT con deteccion automatica de separador
    # y eliminacion de filas basura

    # Deteccion de encode
    with open(file_path, 'rb') as f:
        raw = f.read(10000)
        enc = chardet.detect(raw)['encoding']
    # Lectura con engine Python para flexibilidad y sep=None para deteccion automatica
    try:
        df = pd.read_csv(file_path, encoding=enc, sep=None, engine='python', skip_blank_lines=True)
    
    # Fallback usando latin-1
    except:
        df = pd.read_csv(file_path, encoding='latin-1', sep=r'\s+', engine='python')
    
    # Limpieza de filas con demasiados nulos
    df.dropna(thresh=int(df.shape[1]*0.5), inplace=True)
    # Estandarizacion de nombres (metodo snake_case)
    df.columns = [re.sub(r'[^a-z0-9]', '_', c.lower().strip()) for c in df.columns]
    return df

def standardize_colname(col):
    # Estandarizacion de nombres de columnas (metodo snake_case)
    col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')
    col = col.lower().strip()
    col = re.sub(r'[^a-z0-9]', '_', col)
    col = re.sub(r'_+', '_', col)
    return col

In [10]:
# ## Celda 10 - Obtencion de datos de UEFA

def get_uefa_ranking(year: int) -> pd.DataFrame:
    """
    Obtiene el ranking UEFA para un año dado.
    Primero intenta cargar desde CSV, si no existe lo scrapea.
    """
    csv_path = os.path.join(BASE_DATA_DIR, f"uefa_ranking_{year}.csv")
    
    if os.path.exists(csv_path):
        print(f"Cargando UEFA {year} desde CSV...")
        df = pd.read_csv(csv_path)
        # Asegurar que las columnas existen
        if 'rank' not in df.columns or 'points' not in df.columns:
            print(f"  CSV corrupto para {year}, re-scrapeando...")
            os.remove(csv_path)
            return get_uefa_ranking(year)
        return df
    
    # Scrapear
    print(f"Scrapeando UEFA {year}...")
    url = f"https://kassiesa.net/uefa/data/method5/trank{year}.html"
    html = fetch_html(url)
    if html is None:
        raise ValueError(f"No se pudo obtener UEFA {year}")
    
    df = parse_uefa(html)
    
    # Guardar CSV
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"  Guardado en {csv_path}")
    
    return df

df_uefa_2025 = get_uefa_ranking(2025)
df_uefa_2026 = get_uefa_ranking(2026)
print(f"UEFA 2025: {len(df_uefa_2025)} equipos")
print(f"UEFA 2026: {len(df_uefa_2026)} equipos")

Cargando UEFA 2025 desde CSV...
Cargando UEFA 2026 desde CSV...
UEFA 2025: 170 equipos
UEFA 2026: 176 equipos


In [11]:
# ## Celda 11 - Carga de datos de AFC

def load_afc_ranking(year: int) -> pd.DataFrame:
    # Nombres de archivo: RankingAFC2025.csv, RankingAFC2026.csv
    fname = f"RankingAFC{year}.csv"
    path = os.path.join(BASE_DATA_DIR, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encontró {path}")
    df = pd.read_csv(path)
    # Limpieza
    # Estandarizar nombres
    df.columns = [standardize_colname(c) for c in df.columns]
    # Buscar columnas clave
    rename_map = {}
    for col in df.columns:
        if 'rank' in col:
            rename_map[col] = 'rank'
        elif 'team' in col or 'club' in col:
            rename_map[col] = 'team'
        elif 'country' in col or 'nation' in col:
            rename_map[col] = 'country'
        elif 'point' in col or 'pts' in col or 'score' in col:
            rename_map[col] = 'points'
    df.rename(columns=rename_map, inplace=True)
    # Convertir a numerico
    for col in ['rank', 'points']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['points'])
    return df

df_afc_2025 = load_afc_ranking(2025)
df_afc_2026 = load_afc_ranking(2026)
print(f"AFC 2025: {len(df_afc_2025)} equipos")
print(f"AFC 2026: {len(df_afc_2026)} equipos")

AFC 2025: 16 equipos
AFC 2026: 15 equipos


In [12]:
# ## Celda 12 - Carga de datos CONMEBOL

def load_conmebol() -> pd.DataFrame:
    # Primero intentar cargar CSV si existe
    csv_path = os.path.join(BASE_DATA_DIR, "conmebol_final.csv")
    if os.path.exists(csv_path):
        print("Cargando CONMEBOL desde CSV...")
        df = pd.read_csv(csv_path)
    else:
        # Si no existe CSV, procesar el PDF
        print("Procesando CONMEBOL desde PDF...")
        # Instalar librerías si es necesario
        try:
            import tabula
        except ImportError:
            !pip install tabula-py pdfplumber pypdf2 --quiet
            import tabula
        import pdfplumber

        pdf_path = os.path.join(BASE_DATA_DIR, "Ranking-de-Clubes-CONMEBOL-2026-1.pdf")
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"No se encontró el PDF: {pdf_path}")

        # Intentar con tabula
        try:
            tables = tabula.read_pdf(
                pdf_path,
                pages='all',
                multiple_tables=True,
                guess=True,
                pandas_options={'header': None}
            )
            df_raw = pd.concat(tables, ignore_index=True)
        except Exception as e:
            print(f"tabula falló: {e}. Usando pdfplumber...")
            all_rows = []
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    page_table = page.extract_table()
                    if page_table:
                        clean_page = [[str(cell).replace('\n', ' ').strip() if cell else '' for cell in row] for row in page_table]
                        all_rows.extend(clean_page)
            df_raw = pd.DataFrame(all_rows)

        # Limpieza: asignar primera fila como header si corresponde
        first_row = df_raw.iloc[0].astype(str).str.lower().str.strip().tolist()
        keywords = ['pos', 'equipo', 'pais', 'puntaje', 'total', 'rank', 'team', 'country', 'points', 'club']
        is_header = any(any(kw in val for kw in keywords) for val in first_row)
        if is_header:
            new_headers = df_raw.iloc[0].astype(str).str.strip()
            df_raw.columns = new_headers
            df_raw = df_raw.iloc[1:].reset_index(drop=True)
        # Estandarizar nombres
        df_raw.columns = [standardize_colname(c) for c in df_raw.columns]

        # Identificar columnas
        rename_map = {}
        for col in df_raw.columns:
            if 'pos' in col or 'rank' in col:
                rename_map[col] = 'rank'
            elif 'equipo' in col or 'team' in col:
                rename_map[col] = 'team'
            elif 'pais' in col or 'country' in col or 'pa_s' in col:
                rename_map[col] = 'country'
            elif 'puntaje' in col or 'total' in col or 'points' in col:
                rename_map[col] = 'points'
        df_raw.rename(columns=rename_map, inplace=True)
        # Eliminar columnas no deseadas
        df = df_raw[['rank', 'team', 'country', 'points']].copy()
        # Convertir a numerico
        for col in ['rank', 'points']:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')
        df = df.dropna(subset=['rank', 'points'])
        # Filtrar por paises objetivo y mapear
        df['country'] = df['country'].astype(str).str.strip()
        country_map = {'Argentina': 'ARG', 'Brasil': 'BRA', 'Venezuela': 'VEN'}
        df['country'] = df['country'].map(country_map).fillna(df['country'])
        df = df[df['country'].isin(['ARG', 'BRA', 'VEN'])].copy()
        # Guardar CSV para futuras ejecuciones
        df.to_csv(csv_path, index=False)
        print(f"CONMEBOL guardado en {csv_path}")
    # Asegurar columnas estandar
    df.columns = [standardize_colname(c) for c in df.columns]
    # Si no tiene 'points', buscar columna de puntaje
    if 'points' not in df.columns:
        for col in df.columns:
            if 'total' in col or 'punt' in col or 'score' in col:
                df['points'] = pd.to_numeric(df[col], errors='coerce')
                break
    if 'points' not in df.columns:
        raise ValueError("No se encontró columna de puntos en CONMEBOL")
    df = df.dropna(subset=['rank', 'points'])
    return df

df_conmebol = load_conmebol()
print(f"CONMEBOL: {len(df_conmebol)} equipos")

Cargando CONMEBOL desde CSV...
CONMEBOL: 106 equipos


In [13]:
# ## Celda 13 - Carga de datos ELO 2026

def load_elo_snapshot(fname: str, date_label: str) -> pd.DataFrame:
    path = os.path.join(BASE_DATA_DIR, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encontró {path}")

    # Cargar con load_elo_txt (detecta separador automáticamente)
    df = load_elo_txt(path)

    # --- DETECTAR FORMATO ---
    # Si hay columna 'country' y 'club' separadas : formato Agosto 2025
    # Si solo hay 'club', 'elo', '+/-', 'golo' : formato Septiembre 2026
    
    # 1. Identificar columna de club
    club_col = None
    for col in df.columns:
        if 'club' in col.lower() or 'team' in col.lower():
            club_col = col
            break
    if club_col is None:
        club_col = df.columns[0]  # fallback

    # 2. Identificar columna de ELO
    elo_col = None
    for col in df.columns:
        if 'elo' in col.lower():
            elo_col = col
            break
    if elo_col is None:
        # Buscar columna numérica que no sea rank
        for col in df.columns:
            if col != club_col and pd.api.types.is_numeric_dtype(df[col]):
                elo_col = col
                break
    if elo_col is None:
        raise ValueError("No se encontró columna de ELO")

    # 3. Limpiar nombre del club
    def clean_club_name(val):
        val = str(val).strip()
        # Si el nombre ya está limpio (sin código de país), devolverlo
        if not re.match(r'^[A-Z]{3}\s+\d+\s+', val):
            return val
        # Si tiene código de país y ranking, extraer solo el nombre
        match = re.match(r'^[A-Z]{3}\s+\d+\s+(.+)$', val)
        return match.group(1) if match else val

    df['team_clean'] = df[club_col].apply(clean_club_name)

    # 4. Extraer ranking (si no existe)
    if 'rank' not in df.columns:
        # Si hay columna 'country' con el ranking, extraer de ahí
        if 'country' in df.columns:
            def extract_rank_from_country(val):
                match = re.search(r'\b(\d+)\b', str(val))
                return int(match.group(1)) if match else None
            df['rank'] = df['country'].apply(extract_rank_from_country)
        else:
            # Si no, intentar extraer del nombre compuesto
            def extract_rank(val):
                match = re.search(r'\b(\d+)\b', str(val))
                return int(match.group(1)) if match else None
            df['rank'] = df[club_col].apply(extract_rank)

    # 5. Seleccionar columnas finales
    if 'country' in df.columns:
        # Extraer código de país (3 letras mayúsculas) desde 'country'
        df['country_code'] = df['country'].str.extract(r'^([A-Z]{3})')[0]
    else:
        # Si no hay columna 'country', extraer desde el nombre del club
        df['country_code'] = df[club_col].str.extract(r'^([A-Z]{3})')[0]

    df_final = df[['team_clean', 'country_code', 'rank', elo_col]].rename(
        columns={'team_clean': 'team', 'country_code': 'country', elo_col: 'elo'}
    )
    df_final = df_final.dropna(subset=['elo', 'rank', 'country'])
    df_final['date'] = date_label

    print(f"ELO {date_label}: {len(df_final)} equipos")
    return df_final

df_elo_sep = load_elo_snapshot("ELO20260902.txt", "2026-09-02")

ELO 2026-09-02: 662 equipos


In [14]:
# ## Celda 13B - Carga de datos ELO 2025
fname = "ELO20250802.txt"
path = os.path.join(BASE_DATA_DIR, fname)
df_raw = load_elo_txt(path)
df = df_raw.copy()
df.columns = [c.lower().strip() for c in df.columns]

# Identificar columnas clave
# En este archivo:
# - 'rank' es ranking
# - 'country' es país + ranking (ej. "ENG 1 Liverpool")
# - 'club' es ELO rating 
# - 'elo' es fecha del último partido
club_col = 'club'
elo_col = 'club'          # El rating está en 'club'
rank_col = 'rank'
country_col = 'country'

# Extraer codigo de pais Y nombre del club desde 'country'
def extract_country_and_club(val):
    """
    Extrae el codigo de pais (3 letras) y el nombre del club
    de una cadena con formato: 'ENG 1 Liverpool'
    Retorna: (country_code, club_name)
    """
    val = str(val).strip()
    # Patron: 3 letras mayusculas, espacio, número, espacio, nombre
    match = re.match(r'^([A-Z]{3})\s+\d+\s+(.+)$', val)
    if match:
        return match.group(1), match.group(2)  # (ENG, Liverpool)
    else:
        # Fallback: si no coincide, devolver el valor original y None como país
        return None, val

# Aplicar la extracción
df[['country_code', 'team_clean']] = df[country_col].apply(
    lambda x: pd.Series(extract_country_and_club(x))
)

# Usa 'club' como ELO rating
df['elo'] = pd.to_numeric(df[elo_col], errors='coerce')

# Usar 'rank' existente
df['rank'] = df[rank_col]

# Seleccion de columnas finales
df_final = df[['team_clean', 'country_code', 'rank', 'elo']].rename(
    columns={'team_clean': 'team', 'country_code': 'country'}
)


# Eliminar filas invalidas
df_final = df_final.dropna(subset=['elo', 'rank', 'country'])

# --- 9. Mostrar resultados ---
print(f"   Total equipos: {len(df_final)}")
if len(df_final) > 0:
    print("\n   Primeras 5 filas:")
    print(df_final.head())
    # Guardar en variable global
    df_elo_aug = df_final
    #df_elo_aug['date'] = '2025-08-02'
    print("df_elo_aug creado correctamente con columna 'country'")
else:
    print("No se cargaron equipos. Revisar paso a paso.")

   Total equipos: 100

   Primeras 5 filas:
        team country  rank   elo
0  Liverpool     ENG     1  1993
1    Arsenal     ENG     2  1993
2   Paris SG     FRA     3  1975
3   Man City     ENG     4  1960
4  Barcelona     ESP     5  1945
df_elo_aug creado correctamente con columna 'country'


In [15]:
# ## Celda 14 - Carga de ranking G2Av1513 final

def load_g2a_final(dataset: str):
    # Intentar cargar desde glicko_ratings_final_{dataset}.csv
    path = os.path.join(BASE_DATA_DIR, f"glicko_ratings_final_{dataset}.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"No se encontro {path}")
    df = pd.read_csv(path)
    # Asegurar columnas: 'team', 'mu' (o 'rating'), 'rd', etc.
    df.columns = [standardize_colname(c) for c in df.columns]
    # Buscar columna de equipo y rating
    team_col = None
    mu_col = None
    for col in df.columns:
        if 'team' in col:
            team_col = col
        if 'mu' in col or 'rating' in col:
            mu_col = col
    if team_col is None or mu_col is None:
        raise ValueError("No se encontraron columnas 'team' y 'mu' en G2A final {dataset}")
    df_g2a = df[[team_col, mu_col]].rename(columns={team_col: 'team', mu_col: 'mu'})
    df_g2a = df_g2a.dropna(subset=['mu'])
    # Añadir ranking calculado
    df_g2a['rank'] = df_g2a['mu'].rank(ascending=False, method='min').astype(int)
    return df_g2a

df_g2a_full = load_g2a_final('full')
df_g2a_lite = load_g2a_final('lite')
print(f"G2A Full: {len(df_g2a_full)} equipos")
print(f"G2A Lite: {len(df_g2a_lite)} equipos")

G2A Full: 336 equipos
G2A Lite: 414 equipos


In [16]:
# ## Celda 15 - Carga de Forecasts (Julio 2026)

def load_forecast(dataset: str, forecast_date: str = '2026-07-06') -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    arima_path = os.path.join(BASE_DATA_DIR, f"forecast_arima_52w_{dataset}.csv")
    lstm_path = os.path.join(BASE_DATA_DIR, f"forecast_lstm_52w_{dataset}.csv")
    if not os.path.exists(arima_path) or not os.path.exists(lstm_path):
        raise FileNotFoundError(f"Faltan archivos de forecast para {dataset}")
    df_arima = pd.read_csv(arima_path, index_col=0, parse_dates=True)
    df_lstm = pd.read_csv(lstm_path, index_col=0, parse_dates=True)
    # Buscar fecha mas cercana si no existe
    if forecast_date not in df_arima.index:
        dates = df_arima.index
        idx = dates.get_indexer([pd.Timestamp(forecast_date)], method='nearest')[0]
        forecast_date = dates[idx].strftime('%Y-%m-%d')
        print(f"Usando fecha mas cercana para {dataset}: {forecast_date}")
    arima_week = df_arima.loc[forecast_date].dropna()
    lstm_week = df_lstm.loc[forecast_date].dropna()
    arima_df = pd.DataFrame({'team': arima_week.index, 'mu': arima_week.values})
    lstm_df = pd.DataFrame({'team': lstm_week.index, 'mu': lstm_week.values})
    arima_df['rank'] = arima_df['mu'].rank(ascending=False, method='min').astype(int)
    lstm_df['rank'] = lstm_df['mu'].rank(ascending=False, method='min').astype(int)
    return arima_df, lstm_df, forecast_date

arima_full, lstm_full, _ = load_forecast('full')
arima_lite, lstm_lite, _ = load_forecast('lite')
print(f"ARIMA Full: {len(arima_full)} equipos")
print(f"ARIMA Lite: {len(arima_lite)} equipos")
print(f"LSTM Full: {len(lstm_full)} equipos")
print(f"LSTM Lite: {len(lstm_lite)} equipos")

ARIMA Full: 333 equipos
ARIMA Lite: 411 equipos
LSTM Full: 333 equipos
LSTM Lite: 411 equipos


In [17]:
# %% Celda 16 - MAPEO, DROP Y FILTRO DE PAÍSES UNIFICADO

print("="*70)
print("MAPEO, DROP Y FILTRO DE PAÍSES - UNIFICADO")
print("="*70)

# --- 1. Definir países permitidos ---
PAISES_UEFA = ['ENG', 'ESP', 'GER', 'ITA', 'FRA', 'GRE']
PAISES_ELO_2025 = ['ENG', 'ESP', 'GER', 'ITA', 'FRA', 'GRE']
PAISES_ELO_2026 = ['ENG', 'ESP', 'GER', 'ITA', 'FRA', 'GRE', 
                   'BRA', 'ARG', 'VEN', 'USA', 'MEX', 
                   'JPN', 'KOR', 'KSA', 'IRN']
# CONMEBOL ya está filtrado a ARG, BRA, VEN (no hace falta)

# --- 2. Función para filtrar por país ---
def filter_by_country(df, country_col='country', allowed_countries=None):
    if df is None or len(df) == 0:
        return df
    if allowed_countries is None or country_col not in df.columns:
        return df
    df_copy = df.copy()
    # Normalizar país a mayúsculas
    df_copy[country_col] = df_copy[country_col].astype(str).str.upper().str.strip()
    df_copy = df_copy[df_copy[country_col].isin(allowed_countries)]
    return df_copy

# --- 3. APLICAR FILTRO DE PAÍSES ANTES DEL MAPEO ---
print("\n📊 FILTRANDO POR PAÍS...")

# UEFA
df_uefa_2025 = filter_by_country(df_uefa_2025, 'country', PAISES_UEFA)
df_uefa_2026 = filter_by_country(df_uefa_2026, 'country', PAISES_UEFA)
print(f"UEFA 2025: {len(df_uefa_2025)} equipos (solo {PAISES_UEFA})")
print(f"UEFA 2026: {len(df_uefa_2026)} equipos (solo {PAISES_UEFA})")

# AFC (ya viene filtrado por países asiáticos, pero lo dejamos pasar)
# No aplicamos filtro adicional porque AFC ya tiene solo KSA, JPN, KOR, IRN
print(f"AFC 2025: {len(df_afc_2025)} equipos (ya filtrado)")
print(f"AFC 2026: {len(df_afc_2026)} equipos (ya filtrado)")

# CONMEBOL (ya filtrado a ARG, BRA, VEN)
print(f"CONMEBOL: {len(df_conmebol)} equipos (ya filtrado)")

# ELO 2025
df_elo_aug = filter_by_country(df_elo_aug, 'country', PAISES_ELO_2025)
print(f"ELO Ago 2025: {len(df_elo_aug)} equipos (solo {PAISES_ELO_2025})")

# ELO 2026
df_elo_sep = filter_by_country(df_elo_sep, 'country', PAISES_ELO_2026)
print(f"ELO Sep 2026: {len(df_elo_sep)} equipos (solo {PAISES_ELO_2026})")

# --- 4. TEXTO MANUAL CON MAPEO Y DROP ---
texto_mapeo = """
1899 Hoffenheim - Hoffenheim
AEK - AEK
AS Monaco - Monaco
Aachen - drop
Agropecuario - drop
Al Ahli SFC - AlAhli
Al Hilal - AlHilal
Al Ittihad - AlIttihad
Al Nassr - AlNassr
Al Qadsiah - AlQadsiah
AlAhli
AlHilal
AlIttihad
AlNassr
AlTaawoun
Alaves - Alaves
AlavÃ©s - Alaves
Albacete - drop
Alcorcon - drop
Aldosivi
Almeria
Almirante Brown - drop
AmÃ©rica - AmericaMX
AmÃ©rica Mineiro - AmericaMG
Angers
Annecy - drop
Antequera - drop
Aragua FC - Aragua
Argentinos Juniors - ArgentinosJrs
Aris - Aris
Aris Thessaloniki - Aris
Arsenal - Arsenal
Arsenal de SarandÃ­ - ArsenalSarandi
Ascoli - drop
Aston Villa - AstonVilla
Atalanta
Athletic - AtlParanaense 
Athletic Bilbao - AthBilbao
Atlanta - AtlantaUtd
Atlas
Atletico B - drop
AtlÃ©tico - AtlMadrid
AtlÃ©tico Goianiense - AtlGO
AtlÃ©tico Mineiro - AtleticoMG
AtlÃ©tico Rafaela - drop
AtlÃ©tico San CristÃ³bal - drop
AtlÃ©tico TucumÃ¡n - AtlTucuman
AtlÃ©tico Venezuela - AtlVenezuela
Augsburg
Austin
Auxerre
AvaÃ­ - Avai
Avellino - drop
Bahia 
Banfield
Bangu - drop
Barakaldo - drop
Barcelona
Barcelona B - drop
Bayer Leverkusen - Leverkusen
Bayern
Bayern MÃƒÂ¼nchen - Bayern
Bayern MÃ¼nchen - Bayern
Belgrano
Bielefeld - Arminia
Bilbao - AthBilbao
Birmingham - drop
Blackburn - drop
Bochum
Bologna
Bolton - drop
Borussia Dortmund - Dortmund
Botafogo
Botafogo PB - drop
Botafogo SP - Botafogo
Bournemouth
Bragantino
BrasÃ­lia - drop
Braunschweig - drop
Brentford
Brighton
Brighton & Hove Albion - Brighton
Bristol City - drop
Brusque - drop
Burgos - drop
Burnley
CA Ceuta - drop
CA Los Andes - drop
CE Europa - drop
CR Brasil - drop
CSA AL - CSA
Cadiz
Cagliari
Carabobo
Carabobo FC - Carabobo
Caracas FC - Caracas
Cardiff
Castellon - drop
Catanzaro - drop
CearÃ¡ - Ceara
Celta
Celta de Vigo - Celta
Central CÃ³rdoba - CentralCordobaSdE
Cesena - drop
Chaco For Ever - drop
Chapecoense
Charlotte
Charlton - drop
Chelsea
Cincinnati
Ciudad de BolÃ­var - drop
Club Deportivo PortuguÃ©s - drop
Colegiales de Munro - drop
Columbus - ColumbusCrew
ColÃ³n Santa Fe - ColonSF
ColÃ³n de Santa Fe - ColonSF
Cordoba - drop
Corinthians
Coritiba
Cottbus - drop
Coventry - drop
Cremonese
CriciÃºma - Criciuma
Cruzeiro
Crystal Palace - CrystalPalace
CuiabÃ¡ - Cuiaba
Cultural Leonesa - drop
Daejeon Hana Citizen - DaejeonHanaCitizen
Darmstadt - Darmstadt98
Defensores de Belgrano - drop
Depor - DeporLaCoruna
Deportivo AnzoÃ¡tegui - DepAnzoategui
Deportivo Canarias - drop
Deportivo Galicia - drop
Deportivo La Guaira - DepLaGuaira
Deportivo MorÃ³n - drop
Deportivo Petare - drop
Deportivo TÃ¡chira - DepTachira
Derby - drop
Dijon
Dortmund
Dresden - drop
Dunkerque - drop
DÃ¼sseldorf - Dusseldorf
Eibar
Elche
Eldense - drop
Elversberg - drop
Empoli
Espanyol
Essen - drop
Estudiantes - EstudiantesLP
Estudiantes de Caracas - EstCaracas
Estudiantes de Caseros - drop
Estudiantes de La Plata - EstudiantesLP
Estudiantes de MÃ©rida - EstMerida
Everton
FC Andorra - drop
FC Barcelona - Barcelona
FC La UniÃ³n AtlÃ©tico - drop
FC Machida Zelvia - MachidaZelvia
FRA FC Fleury 91 - drop
FRA Le Puy Foot 43 - drop
FRA Paris 13 Atletico - drop
FSV Mainz 05 - Mainz
Ferro - drop
Ferro Carril Oeste - drop
Ferrocarril Midland - drop
FerroviÃ¡ria SP - drop
Figueirense - drop
Fiorentina
Flamengo
Fluminense
Forest - NottsForest
Fortaleza
Fortaleza EC - Fortaleza
Frankfurt - Eintracht
Freiburg
Frosinone
Fulham
FÃ¼rth - drop
GE Jujuy - drop
GER MÃ¼nchen 60 - drop
GambaOsaka 
Gangwon
Genoa - drop
Getafe
Gijon - drop
Gimnasia LP - GimnasiaYEsgrimaLP
Gimnasia La Plata - GimnasiaYEsgrimaLP
Gimnasia y Tiro de Salta - drop
Girona
Girona FC - Girona
GoiÃ¡s - Goias
Granada
Grenoble - drop
GrÃªmio - Gremio
GrÃªmio Barueri - drop
Guadalajara
Guarani - drop
Guarani FC - drop
GyE Mendoza - drop
Hamburg
Hannover
Heidenheim
Hermanos ColmenÃ¡rez - InterBarinas
Hertha
Hoffenheim
Holstein - HolsteinKiel
Huesca
Hull - drop
HuracÃ¡n - Huracan
ITA Calcio Lecco 1912 - drop
ITA US Pergolettese 1932 - drop
Ind Rivadavia - IndependienteRivadavia
Ipswich
Jeonbuk FC - JeonbukHyundai
JeounbukHyundai - JeonbukHyundai
Joinville - drop
Juve Stabia - drop
Juventud Torremolinos - drop
Juventude
Juventus
Jwangju - Gwangju
KOR Bucheon FC 1995 - drop
Karlsruhe - drop
KawasakiF - KawasakiFrontale
Kyoto Purple Sanga - KyotoSanga
KÃ¶ln - Koln
LAFC
LanÃºs - Lanus
Lara FC - drop
Lautern - drop
Lazio
Le Mans - drop
Lecce
Leeds - LeedsUtd
Leganes
Leicester
Levante
Leverkusen
LeÃ³n - Leon
Lille
Lille OSC - Lille
Lincoln City - drop
Liverpool
Llaneros de Guanare - Llaneros
Londrina - drop
Lorient
Luton
MachidaZelvia
Madryn - drop
Magdeburg - drop
Mainz
MaipÃº - drop
Malaga
Man City - ManCity
Man United - ManUtd
Manchester City - ManCity
Manchester United - ManUtd
Mantova - drop
Maranhao - drop
MaringÃ¡ PR - drop
MarÃ­timo - drop
Metropolitanos
Metz
Middlesbrough - drop
Milan
Millwall - drop
Mineros de Guayana - Mineros
MinervÃ©n - drop
Mirandes - drop
Mirassol
Modena - drop
Monaco
Monagas
Monterrey
Montpellier
Monza
Murcia - drop
NYC FC - NYCFC
Nagoya Grampus - NagoyaGrampus
Nantes
Napoli
Nashville
Necaxa
New York RB - NYRB
Newcastle
Newcastle United - Newcastle
Nice
Norwich
Nottingham Forest - NottsForest
Novorizontino - drop
Numancia - drop
NÃ¡utico - drop
NÃ¼rnberg - Nuremberg
OGC Nice - Nice
Olympiakos - Olympiacos
Olympiakos Piraeus - Olympiacos
Olympique Lyon - OLyon
Olympique Marseille - OMarseille
OperÃ¡rio - drop
Osasuna
OsnabrÃ¼ck - drop
Ourense - drop
PAOK
PAOK Thessaloniki - PAOK
Pachuca
Paderborn
Palermo - drop
Palmeiras
Panathinaikos
ParanÃ¡ - Parana
Paris SG - PSG
Parma
Patronato
Paulista - drop
Paysandu - drop
Pepeganga - drop
Persepolis
Philadelphia - PhiladelphiaUnion
Pisa - drop
Platense
Plymouth - drop
PohangSteelers
Ponferradina - drop
Pontevedra - drop
Portsmouth - drop
Portuguesa
Portuguesa FC (Venezuela) - Portuguesa
Preston - drop
QPR - drop
QuerÃ©taro - Queretaro
Quilmes - drop
RC Celta B - drop
RC Lens - RacingLens
RC Strasbourg - RacingStrasbourg 
Racing de CÃ³rdoba - drop
Real Madrid B - drop
Real Sociedad B - drop
Red Bull Bragantino - Bragantino
Red Star - drop
Remo - drop
Riestra - drop
Rodez - drop
Roma
Rostock - drop
RÃ­o Cuarto - drop
SC Freiburg - Freiburg
SM Tucuman - SanMartinTucuman
Sabadell - drop
San Jose - SJEarthquakes
San Juan - drop
San Luis - drop
San Miguel - drop
SanfrecceHiroshima
Sant Andreu - drop
Santa Cruz - drop
Santander - drop
Santo AndrÃ© - drop
Sassuolo
Schalke - Schalke04
Seoul
Sepahan
Sevilla
Sheffield United - SheffieldUtd
Southampton
Spezia
Sport - drop
Stade Brestois - StadeBrestois29 
Stade Rennais - StadeRennais
Stockport - drop
Stoke
Stuttgart
Sunderland - drop
Swansea
SÃ£o Bernardo - drop
SÃ£o Caetano - drop
SÃ£o Paulo - SaoPaulo
Talleres de CÃ³rdoba - TalleresCordoba
Tarragona - drop
Temperley
Tenerife - drop
Tijuana
Toluca
Torino
Tottenham - Spurs
Tottenham Hotspur - Spurs
Toulouse
TractorSazi
Tristan SuÃ¡rez - drop
Troyes
Trujillanos
UD Ibiza - drop
Udinese
Ulsan - UlsanHD
UlsanHD - UlsanHD
UlsanHyundai - UlsanHD
Union Brescia - drop
Unionistas - drop
Universidad Los Andes MÃ©rida - drop
UniÃ³n AtlÃ©tico Maracaibo - drop
UniÃ³n Santa Fe - UnionSantaFe
UniÃ³n de Santa Fe - UnionSantaFe 
Valencia
Valencia FÃºtbol Club - Valencia
Vancouver - VancWhitecaps
Venezia
Verl - drop
Verona - drop
VfB Stuttgart - Stuttgart
Vicenza - drop
Vila Nova GO - drop
Villarreal
Villarreal B - drop
VisselKobe
VitÃ³ria - drop
VÃ©lez Sarsfield - VelezSarsfield
Watford
Werder - WerderBremen
West Brom - WestBrom
West Ham - WestHam
West Ham United - WestHam
Wolfsburg
Wolves - Wolverhampton
Wrexham - drop
Yaracuyanos
YokofamaFMarinos - YokohamaMarinos
Zamora
Zamora FC - Zamora
Zaragoza - drop
Zulia
KawasakiFrontal - KawasakiFrontale
Atlético Mineiro - AtleticoMG
São Paulo - SaoPaulo
Grêmio - Gremio
Lanús - Lanus
Vélez Sarsfield - VelezSarsfield
Alavés - Alaves
America
AmericaMX - America
Ολυμπιακός - Olympiacos
Talleres de Córdoba - TalleresCordoba
Deportivo Táchira - DepTachira
DEpTachira - DepTachira
Huracán - Huracan
Colón Santa Fe - Colon
ColonSantaFe - Colon
Independiente Rivadavia - IndependienteRivadavia
América - America
Criciúma - Criciuma
Atlético Goianiense - AtleticoGO
Ceará - Ceara
Central Córdoba - CentralCordobaSdE
América Mineiro - AmericaMG
Arsenal de Sarandí - ArsenalSarandi
Cuiabá - Cuiaba
Unión Santa Fe - UnionSantaFe
Goiás - Goias
Estudiantes de Mérida - EstMerida
Atlético Tucumán - AtlTucuman
São Caetano - drop
Vitória - drop
Minervén - drop
Universidad Los Andes Mérida - drop
Atlético San Cristóbal - drop
Unión Atlético Maracaibo - drop
Marítimo - drop
Club Deportivo Portugués - drop
Köln - Koln
Náutico - drop
Deportivo Morón - drop
Tristan Suárez - drop
Nürnberg - Nuremberg
Operário - drop
São Bernardo - drop
Avaí - Avai
Maipú - drop
Querétaro - Queretaro
Colón de Santa Fe - Colon
Ferroviária SP - drop
AtlGO - AtleticoGO
Valencia Fútbol Club - drop
Paraná - Parana
Santo André - drop
Deportivo Anzoátegui - DepAnzoategui
Hermanos Colmenárez - InterBarinas
Atlético Venezuela - AtlVenezuela
Brasília - drop
Grêmio Barueri - drop
Unión de Santa Fe - UnionSantaFe
Río Cuarto - drop
León - Leon
Düsseldorf - Dusseldorf
Osnabrück - drop
Ciudad de Bolívar - drop
Maringá PR - drop
Atlético Rafaela - drop
Fürth - drop
Racing de Córdoba - drop
FC La Unión Atlético - drop
Grenoble - drop
GER München 60 - drop
Atlético - AtlMadrid
Bayern München - Bayern
Inter de Limeira - drop
ac milan - Milan
aek athens - AEK
as roma - Roma
ath paranaense - AtlParanaense
athletic club - AthBilbao
athletico paranaense - AtlParanaense
atl goianiense - AtleticoGO
atlÃ©tico madrid - AtlMadrid
avispa fukuoka - AvispaFukuoka
barracas central - BarracasCentral
betis - RealBetis
boca juniors - BocaJuniors
brest - StadeBrestois29
cerezo osaka - CerezoOsaka
chicago fire - ChicagoFire
colorado rapids - ColRapids
como - Como1907
cruz azul - CruzAzul
defensa y justicia - DefensaYJusticia
deportivo lara - DepLara
eintracht frankfurt - Eintracht
esteghlal - EsteghlalTehran
fagiano okayama - FagianoOkayama
fc dallas - Dallas
fc seoul - Seoul
fc tokyo - FCTokyo
gamba osaka - GambaOsaka
gangwon fc - Gangwon
gimcheon sangmu - GimcheonSangmu
gladbach - MGladbach
godoy cruz - GodoyCruz
guingamp - EnAvantGuingamp
houston dynamo - HouDynamo
incheon united - IncheonUtd
independiente - IndependienteAvellaneda
instituto - InstitutoCordoba 
inter - InterPA
inter miami - InterMiami
internacional - InterPA
internazionale - InterMilan
kashima antlers - KashimaAntlers
kashiwa reysol - KashiwaReysol
kawasaki frontale - KawasakiFrontale
la galaxy - LAGalaxy
las palmas - LasPalmas
le havre - LeHavre
lens - RacingLens
lyon - OLyon
mallorca - RealMallorca
marseille - OMarseille
minnesota utd - MinnesotaUtd
ne revolution - NERevolution
newell s old boys - NewellsOldBoys
orlando city - OrlandoCity
oviedo - drop
paris fc - drop
paris saint germain - PSG
pisa - drop
pohang steelers - PohangSteelers 
ponte preta - drop
portland timbers - PortTimbers
puerto cabello - AcPtoCabello
racing club - RacingAvellaneda
rayo vallecano - RayoVallecano
rayo zuliano - Zulia
rb leipzig - Leipzig
real betis - RealBetis
real madrid - RealMadrid
real salt lake - RealSaltLake
real sociedad - RealSociedad
reims - StadeReims
rennes - StadeRennais
river plate - RiverPlate
rosario central - RosarioCentral
saint etienne - SaintEtienne
san diego fc - SanDiego
san lorenzo - SanLorenzo
sanfrecce hiroshima - SanfrecceHiroshima
santos - SantosBR
sarmiento - SarmientoJunin
seattle sounders - SeaSounders
shimizu s pulse - ShimizuSPulse
st louis city - StLouisCity
st pauli - StPauli
strasbourg - RacingStrasbourg
talleres - TalleresCordoba
tigre - TigreBA
toronto fc - Toronto
tractor sazi - TractorSazi
uanl - TigresUANL
unam - PumasUNAM
union berlin - UnionBerlin
universidad central - UCVFC
urawa reds - UrawaRedDiamonds
valladolid - RealValladolid
vasco da gama - VascoDaGama
vissel kobe - VisselKobe
yokohama f marinos - YokohamaMarinos
zamora - Zamora
paris saint-germain - PSG
AthParanaense - AtlParanaense
newell's old boys - NewellsOldBoys
saint-etienne - SaintEtienne
st. louis city - StLouisCity
yokohama f. marinos - YokohamaMarinos
shimizu s-pulse - ShimizuSPulse
"""

# --- 5. Procesar el texto ---
manual_mapping = {}
drop_list = []

lines = texto_mapeo.strip().split('\n')
for line in lines:
    line = line.strip()
    if not line:
        continue
    
    if ' - ' in line:
        original, normalized = line.split(' - ', 1)
        original = original.strip()
        normalized = normalized.strip()
        
        if normalized.lower() == 'drop':
            drop_list.append(original)
        else:
            manual_mapping[original] = normalized
    else:
        original = line.strip()
        if original and original not in manual_mapping:
            manual_mapping[original] = original

print(f"\n✅ Mapeo: {len(manual_mapping)} entradas")
print(f"✅ Drop list: {len(drop_list)} equipos a eliminar")

# --- 6. Función unificada: mapear + dropear ---
def apply_mapping_and_drop(df, team_col='team'):
    df_copy = df.copy()
    
    # 1. Convertir nombres a minúsculas
    df_copy[team_col] = df_copy[team_col].astype(str).str.strip().str.lower()
    
    # 2. Crear mapeo con claves en minúsculas
    mapping_lower = {k.lower(): v for k, v in manual_mapping.items()}
    drop_lower = [d.lower() for d in drop_list]
    
    # 3. Aplicar mapeo
    df_copy[team_col] = df_copy[team_col].map(lambda x: mapping_lower.get(x, x))
    
    # 4. Eliminar drops
    df_copy = df_copy[~df_copy[team_col].isin(drop_lower)]
    
    return df_copy

# --- 7. APLICAR MAPEO Y DROP (ya filtrados por país) ---
print("\n" + "="*70)
print("APLICANDO MAPEO Y DROP")
print("="*70)

df_uefa_2025 = apply_mapping_and_drop(df_uefa_2025)
df_uefa_2026 = apply_mapping_and_drop(df_uefa_2026)
print(f"UEFA 2025: {len(df_uefa_2025)} equipos")
print(f"UEFA 2026: {len(df_uefa_2026)} equipos")

df_afc_2025 = apply_mapping_and_drop(df_afc_2025)
df_afc_2026 = apply_mapping_and_drop(df_afc_2026)
print(f"AFC 2025: {len(df_afc_2025)} equipos")
print(f"AFC 2026: {len(df_afc_2026)} equipos")

df_conmebol = apply_mapping_and_drop(df_conmebol)
print(f"CONMEBOL: {len(df_conmebol)} equipos")

df_elo_aug = apply_mapping_and_drop(df_elo_aug)
df_elo_sep = apply_mapping_and_drop(df_elo_sep)
print(f"ELO Ago 2025: {len(df_elo_aug)} equipos")
print(f"ELO Sep 2026: {len(df_elo_sep)} equipos")

print("\n✅ Filtro de países, mapeo y DROP aplicados correctamente.")

MAPEO, DROP Y FILTRO DE PAÍSES - UNIFICADO

📊 FILTRANDO POR PAÍS...
UEFA 2025: 44 equipos (solo ['ENG', 'ESP', 'GER', 'ITA', 'FRA', 'GRE'])
UEFA 2026: 47 equipos (solo ['ENG', 'ESP', 'GER', 'ITA', 'FRA', 'GRE'])
AFC 2025: 16 equipos (ya filtrado)
AFC 2026: 15 equipos (ya filtrado)
CONMEBOL: 106 equipos (ya filtrado)
ELO Ago 2025: 80 equipos (solo ['ENG', 'ESP', 'GER', 'ITA', 'FRA', 'GRE'])
ELO Sep 2026: 412 equipos (solo ['ENG', 'ESP', 'GER', 'ITA', 'FRA', 'GRE', 'BRA', 'ARG', 'VEN', 'USA', 'MEX', 'JPN', 'KOR', 'KSA', 'IRN'])

✅ Mapeo: 422 entradas
✅ Drop list: 190 equipos a eliminar

APLICANDO MAPEO Y DROP
UEFA 2025: 44 equipos
UEFA 2026: 47 equipos
AFC 2025: 16 equipos
AFC 2026: 15 equipos
CONMEBOL: 77 equipos
ELO Ago 2025: 79 equipos
ELO Sep 2026: 276 equipos

✅ Filtro de países, mapeo y DROP aplicados correctamente.


In [18]:
# %% Celda 17 - VERIFICACIÓN DE MAPEO Y DROP (CON final_mapping)

print("="*70)
print("VERIFICACIÓN DE MAPEO Y DROP")
print("="*70)

# --- 1. Verificar que ningún drop haya quedado ---
print("\n🔍 Verificando que NO queden equipos en drop_list:")

drops_found_total = 0
for name, df in [
    ('UEFA 2025', df_uefa_2025),
    ('UEFA 2026', df_uefa_2026),
    ('AFC 2025', df_afc_2025),
    ('AFC 2026', df_afc_2026),
    ('CONMEBOL', df_conmebol),
    ('ELO Ago 2025', df_elo_aug),
    ('ELO Sep 2026', df_elo_sep),
]:
    if df is not None and len(df) > 0:
        teams = set(df['team'].astype(str).str.strip().unique())
        drops_found = teams.intersection(set(drop_list))
        if drops_found:
            print(f"   ⚠️  {name}: {len(drops_found)} drops encontrados: {sorted(drops_found)[:10]}...")
            drops_found_total += len(drops_found)
        else:
            print(f"   ✅ {name}: sin drops")

if drops_found_total == 0:
    print("\n✅ TODOS los drops fueron eliminados correctamente.")
else:
    print(f"\n⚠️  Quedan {drops_found_total} equipos en drop_list. Revisar mapeo.")

# --- 2. Verificar nombres NO mapeados (usando manual_mapping) ---
print("\n🔍 Verificando nombres NO mapeados por fuente:")

def check_unmapped(df, label):
    if df is None or len(df) == 0:
        return []
    names = df['team'].astype(str).str.strip().unique()
    # Nombres que NO están en G2A
    g2a_names = set(df_g2a_lite['team'].astype(str).str.strip().unique())
    unmapped = [n for n in names if n not in g2a_names]
    return unmapped

total_unmapped = 0
all_unmapped = set()
for name, df in [
    ('UEFA 2025', df_uefa_2025),
    ('UEFA 2026', df_uefa_2026),
    ('AFC 2025', df_afc_2025),
    ('AFC 2026', df_afc_2026),
    ('CONMEBOL', df_conmebol),
    ('ELO Ago 2025', df_elo_aug),
    ('ELO Sep 2026', df_elo_sep),
]:
    unmapped = check_unmapped(df, name)
    if unmapped:
        print(f"   ⚠️  {name}: {len(unmapped)} no mapeados")
        if len(unmapped) <= 10:
            print(f"      {unmapped}")
        else:
            print(f"      Primeros 10: {unmapped[:10]}...")
        total_unmapped += len(unmapped)
        all_unmapped.update(unmapped)
    else:
        if df is not None and len(df) > 0:
            print(f"   ✅ {name}: todos mapeados")

if total_unmapped == 0:
    print("\n✅ TODOS los nombres están mapeados.")
else:
    print(f"\n⚠️  Hay {total_unmapped} nombres sin mapear.")
    if all_unmapped:
        df_unmapped = pd.DataFrame(sorted(all_unmapped), columns=['nombre_original'])
        output_path = os.path.join(BASE_DATA_DIR, "nombres_faltantes_en_mapeo.csv")
        df_unmapped.to_csv(output_path, index=False)
        print(f"📁 Lista guardada en: {output_path}")

# --- 3. Resumen final de tamaños ---
print("\n" + "="*70)
print("RESUMEN FINAL - TAMAÑOS DE DATAFRAMES")
print("="*70)

summary = []
for name, df in [
    ('G2A Lite', df_g2a_lite),
    ('G2A Full', df_g2a_full),
    ('UEFA 2025', df_uefa_2025),
    ('UEFA 2026', df_uefa_2026),
    ('AFC 2025', df_afc_2025),
    ('AFC 2026', df_afc_2026),
    ('CONMEBOL', df_conmebol),
    ('ELO Ago 2025', df_elo_aug),
    ('ELO Sep 2026', df_elo_sep),
]:
    if df is not None:
        summary.append({'Fuente': name, 'Equipos': len(df)})

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

print("\n✅ VERIFICACIÓN COMPLETADA.")

VERIFICACIÓN DE MAPEO Y DROP

🔍 Verificando que NO queden equipos en drop_list:
   ✅ UEFA 2025: sin drops
   ✅ UEFA 2026: sin drops
   ✅ AFC 2025: sin drops
   ✅ AFC 2026: sin drops
   ✅ CONMEBOL: sin drops
   ✅ ELO Ago 2025: sin drops
   ✅ ELO Sep 2026: sin drops

✅ TODOS los drops fueron eliminados correctamente.

🔍 Verificando nombres NO mapeados por fuente:
   ✅ UEFA 2025: todos mapeados
   ✅ UEFA 2026: todos mapeados
   ✅ AFC 2025: todos mapeados
   ✅ AFC 2026: todos mapeados
   ✅ CONMEBOL: todos mapeados
   ✅ ELO Ago 2025: todos mapeados
   ⚠️  ELO Sep 2026: 1 no mapeados
      ['inter de limeira - sp']

⚠️  Hay 1 nombres sin mapear.
📁 Lista guardada en: /content/drive/MyDrive/TEG02-2025Arellano/Data/nombres_faltantes_en_mapeo.csv

RESUMEN FINAL - TAMAÑOS DE DATAFRAMES
      Fuente  Equipos
    G2A Lite      414
    G2A Full      336
   UEFA 2025       44
   UEFA 2026       47
    AFC 2025       16
    AFC 2026       15
    CONMEBOL       77
ELO Ago 2025       79
ELO Sep 2026    

In [19]:
# %% Celda 18 - VALIDACIONES: CORRELACIÓN DE RANKINGS (VERSIÓN DEFINITIVA)

print("="*70)
print("VALIDACIONES - CORRELACIÓN DE RANKINGS")
print("="*70)

# --- 1. Función para alinear y calcular métricas ---
def align_and_compare(df_interno, df_externo, 
                      col_interno='mu', col_externo='points',
                      label='', verbose=True):
    """
    Alinea dos DataFrames por nombre de equipo (todo en minúsculas)
    y calcula métricas de concordancia.
    """
    # Copias
    df_int = df_interno.copy()
    df_ext = df_externo.copy()
    
    # Asegurar columna 'team'
    if 'team' not in df_int.columns:
        for col in df_int.columns:
            if 'team' in col.lower() or 'club' in col.lower():
                df_int.rename(columns={col: 'team'}, inplace=True)
                break
    if 'team' not in df_ext.columns:
        for col in df_ext.columns:
            if 'team' in col.lower() or 'club' in col.lower():
                df_ext.rename(columns={col: 'team'}, inplace=True)
                break
    
    # Buscar columna de score en interno si no se encuentra
    if col_interno not in df_int.columns:
        for col in df_int.columns:
            if any(k in col.lower() for k in ['mu', 'rating', 'score']):
                df_int.rename(columns={col: col_interno}, inplace=True)
                break
    
    # Buscar columna de score en externo si no se encuentra
    if col_externo not in df_ext.columns:
        for col in df_ext.columns:
            if any(k in col.lower() for k in ['points', 'elo', 'score', 'puntos']):
                df_ext.rename(columns={col: col_externo}, inplace=True)
                break
    
    # Verificar que ambas columnas existen
    if col_interno not in df_int.columns:
        if verbose:
            print(f"   ⚠️  {label}: columna interna '{col_interno}' no encontrada. Columnas: {df_int.columns.tolist()}")
        return {'label': label, 'n': 0, 'rho': np.nan, 'tau': np.nan,
                'mae_pos': np.nan, 'overlap10': np.nan, 'overlap20': np.nan}
    if col_externo not in df_ext.columns:
        if verbose:
            print(f"   ⚠️  {label}: columna externa '{col_externo}' no encontrada. Columnas: {df_ext.columns.tolist()}")
        return {'label': label, 'n': 0, 'rho': np.nan, 'tau': np.nan,
                'mae_pos': np.nan, 'overlap10': np.nan, 'overlap20': np.nan}
    
    # Convertir nombres de equipo a minúsculas
    df_int['team'] = df_int['team'].astype(str).str.strip().str.lower()
    df_ext['team'] = df_ext['team'].astype(str).str.strip().str.lower()
    
    # Renombrar columnas de score ANTES del merge para evitar sufijos
    df_int.rename(columns={col_interno: 'score_int'}, inplace=True)
    df_ext.rename(columns={col_externo: 'score_ext'}, inplace=True)
    
    # Seleccionar solo columnas necesarias
    df_int_subset = df_int[['team', 'score_int']].copy()
    df_ext_subset = df_ext[['team', 'score_ext']].copy()
    
    # Unir por equipo (inner join)
    merged = pd.merge(df_int_subset, df_ext_subset, on='team', how='inner')
    n = len(merged)
    
    if n < 3:
        if verbose:
            print(f"   ⚠️  {label}: solo {n} equipos en común (mínimo 3)")
        return {
            'label': label, 'n': n, 'rho': np.nan, 'tau': np.nan,
            'mae_pos': np.nan, 'overlap10': np.nan, 'overlap20': np.nan
        }
    
    # Calcular rankings
    merged['rank_int'] = merged['score_int'].rank(ascending=False, method='min')
    merged['rank_ext'] = merged['score_ext'].rank(ascending=False, method='min')
    
    # Métricas
    rho, _ = spearmanr(merged['rank_int'], merged['rank_ext'])
    tau, _ = kendalltau(merged['rank_int'], merged['rank_ext'])
    mae_pos = mean_absolute_error(merged['rank_int'], merged['rank_ext'])
    
    # Overlap@K
    top10_int = set(merged.nsmallest(10, 'rank_int')['team'])
    top10_ext = set(merged.nsmallest(10, 'rank_ext')['team'])
    overlap10 = len(top10_int & top10_ext) / 10
    
    top20_int = set(merged.nsmallest(20, 'rank_int')['team'])
    top20_ext = set(merged.nsmallest(20, 'rank_ext')['team'])
    overlap20 = len(top20_int & top20_ext) / 20
    
    if verbose:
        print(f"   ✅ {label}: n={n}, ρ={rho:.4f}, τ={tau:.4f}, O10={overlap10:.3f}, O20={overlap20:.3f}, MAE={mae_pos:.1f}")
    
    return {
        'label': label, 'n': n, 'rho': rho, 'tau': tau,
        'mae_pos': mae_pos, 'overlap10': overlap10, 'overlap20': overlap20,
        'merged': merged
    }

# --- 2. Definir comparaciones ---
comparisons = []
df_ref = df_g2a_lite
# Bloque 1: Europa 2025
comparisons.append(('UEFA 2025', df_ref, df_uefa_2025, 'mu', 'points'))
comparisons.append(('ELO Ago 2025', df_ref, df_elo_aug, 'mu', 'elo'))

# Bloque 2: Asia 2025
comparisons.append(('AFC 2025', df_ref, df_afc_2025, 'mu', 'points'))

# Bloque 3: Sudamérica 2026
comparisons.append(('CONMEBOL 2026', df_ref, df_conmebol, 'mu', 'points'))

# Bloque 4: Proyecciones 2026 (cuando ARIMA/LSTM estén listos)
if 'lstm_lite' in globals() and lstm_lite is not None and len(lstm_lite) > 0:
    comparisons.append(('LSTM vs UEFA 2026', lstm_lite, df_uefa_2026, 'mu', 'points'))
    comparisons.append(('LSTM vs ELO Sep 2026', lstm_lite, df_elo_sep, 'mu', 'elo'))
    comparisons.append(('LSTM vs AFC 2026', lstm_lite, df_afc_2026, 'mu', 'points'))

# Opcional: ARIMA/LSTM
if 'arima_lite' in globals() and arima_lite is not None and len(arima_lite) > 0:
    comparisons.append(('ARIMA Lite', df_ref, arima_lite, 'mu', 'mu'))
if 'lstm_lite' in globals() and lstm_lite is not None and len(lstm_lite) > 0:
    comparisons.append(('LSTM Lite', df_ref, lstm_lite, 'mu', 'mu'))

print(f"\n📋 Ejecutando {len(comparisons)} comparaciones...\n")

# --- 3. Ejecutar ---
results = []
for label, df_int, df_ext, col_int, col_ext in comparisons:
    res = align_and_compare(df_int, df_ext, col_int, col_ext, label)
    results.append(res)

# --- 4. DataFrame de resultados ---
df_results = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'merged'} for r in results
])

print("\n" + "="*70)
print("RESUMEN DE VALIDACIONES")
print("="*70)
print(df_results[['label', 'n', 'rho', 'tau', 'overlap10', 'overlap20', 'mae_pos']].to_string(index=False))

# --- 5. Tabla LaTeX ---
latex_table = "\\begin{table}[htbp]\n\\centering\n\\caption{Validaciones contra rankings institucionales (Lite)}\n\\label{tab:validaciones_lite}\n\\begin{tabular}{lcccccc}\n\\toprule\n\\textbf{Validación} & \\textbf{N} & \\textbf{Spearman $\\rho$} & \\textbf{Kendall $\\tau$} & \\textbf{Overlap@10} & \\textbf{Overlap@20} & \\textbf{MAE pos} \\\\\n\\midrule\n"
for _, row in df_results.iterrows():
    label = row['label']
    n = int(row['n']) if not pd.isna(row['n']) else 0
    rho = f"{row['rho']:.3f}" if not pd.isna(row['rho']) else "—"
    tau = f"{row['tau']:.3f}" if not pd.isna(row['tau']) else "—"
    ov10 = f"{row['overlap10']:.3f}" if not pd.isna(row['overlap10']) else "—"
    ov20 = f"{row['overlap20']:.3f}" if not pd.isna(row['overlap20']) else "—"
    mae = f"{row['mae_pos']:.1f}" if not pd.isna(row['mae_pos']) else "—"
    latex_table += f"{label} & {n} & {rho} & {tau} & {ov10} & {ov20} & {mae} \\\\\n"
latex_table += "\\bottomrule\n\\end{tabular}\n\\end{table}"

print("\n" + "="*70)
print("TABLA LATEX GENERADA")
print("="*70)
print(latex_table)

# Guardar
with open(os.path.join(BASE_DATA_DIR, "validaciones_lite.tex"), "w") as f:
    f.write(latex_table)
print(f"\n✅ Tabla guardada en: {os.path.join(BASE_DATA_DIR, 'validaciones_lite.tex')}")

# --- 6. Scatter plots (opcional) ---
print("\n📊 Generando scatter plots...")
for res in results:
    if res['n'] < 5 or np.isnan(res['rho']):
        continue
    merged = res['merged']
    label = res['label'].replace(' ', '_')
    fig, ax = plt.subplots(figsize=(6,6))
    ax.scatter(merged['rank_int'], merged['rank_ext'], alpha=0.5, s=20)
    ax.plot([0, max(merged['rank_int'])], [0, max(merged['rank_int'])], 'r--', alpha=0.7)
    ax.set_xlabel('Ranking G2A')
    ax.set_ylabel('Ranking externo')
    ax.set_title(f'{label} (ρ={res["rho"]:.3f}, n={res["n"]})')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DATA_DIR, f"scatter_{label}.png"), dpi=150)
    plt.close()
    print(f"   ✅ Scatter: scatter_{label}.png")

print("\n✅ VALIDACIONES COMPLETADAS.")

VALIDACIONES - CORRELACIÓN DE RANKINGS

📋 Ejecutando 9 comparaciones...

   ✅ UEFA 2025: n=44, ρ=0.6506, τ=0.4749, O10=0.700, O20=0.750, MAE=8.1
   ✅ ELO Ago 2025: n=79, ρ=0.7863, τ=0.6089, O10=0.700, O20=0.750, MAE=11.4
   ✅ AFC 2025: n=16, ρ=0.0794, τ=0.0500, O10=0.600, O20=0.800, MAE=5.1
   ✅ CONMEBOL 2026: n=77, ρ=0.6433, τ=0.4699, O10=0.600, O20=0.600, MAE=14.0
   ✅ LSTM vs UEFA 2026: n=47, ρ=0.7039, τ=0.5393, O10=0.900, O20=0.700, MAE=7.5
   ✅ LSTM vs ELO Sep 2026: n=273, ρ=0.4731, τ=0.3331, O10=0.700, O20=0.650, MAE=61.3
   ✅ LSTM vs AFC 2026: n=15, ρ=0.1698, τ=0.1148, O10=0.800, O20=0.750, MAE=4.9
   ✅ ARIMA Lite: n=411, ρ=0.9902, τ=0.9411, O10=0.800, O20=0.950, MAE=9.2
   ✅ LSTM Lite: n=411, ρ=0.9847, τ=0.9031, O10=0.900, O20=0.950, MAE=14.2

RESUMEN DE VALIDACIONES
               label   n      rho      tau  overlap10  overlap20   mae_pos
           UEFA 2025  44 0.650622 0.474881        0.7       0.75  8.068182
        ELO Ago 2025  79 0.786268 0.608936        0.7       0.75

In [20]:
# %% Celda 19 - TOP 20 GLOBAL DE CADA RANKING

print("="*70)
print("TOP 20 GLOBAL DE CADA RANKING")
print("="*70)

# --- Definir fuentes y sus columnas de score ---
fuentes = {}

# G2A Lite
if df_g2a_lite is not None and len(df_g2a_lite) > 0:
    fuentes['G2A Lite'] = (df_g2a_lite, 'mu')

# G2A Full
if df_g2a_full is not None and len(df_g2a_full) > 0:
    fuentes['G2A Full'] = (df_g2a_full, 'mu')

# LSTM Lite
if 'lstm_lite' in globals() and lstm_lite is not None and len(lstm_lite) > 0:
    fuentes['LSTM Lite'] = (lstm_lite, 'mu')

# LSTM Full
if 'lstm_full' in globals() and lstm_full is not None and len(lstm_lite) > 0:
    fuentes['LSTM Full'] = (lstm_full, 'mu')

# ARIMA Lite (si existe)
if 'arima_lite' in globals() and arima_lite is not None and len(arima_lite) > 0:
    fuentes['ARIMA Lite'] = (arima_lite, 'mu')

# ARIMA Full (si existe)
if 'arima_full' in globals() and arima_full is not None and len(arima_lite) > 0:
    fuentes['ARIMA Full'] = (arima_full, 'mu')

# UEFA 2025
if df_uefa_2025 is not None and len(df_uefa_2025) > 0:
    fuentes['UEFA 2025'] = (df_uefa_2025, 'points')

# UEFA 2026
if df_uefa_2026 is not None and len(df_uefa_2026) > 0:
    fuentes['UEFA 2026'] = (df_uefa_2026, 'points')

# ELO Ago 2025
if df_elo_aug is not None and len(df_elo_aug) > 0:
    fuentes['ELO Ago 2025'] = (df_elo_aug, 'elo')

# ELO Sep 2026
if df_elo_sep is not None and len(df_elo_sep) > 0:
    fuentes['ELO Sep 2026'] = (df_elo_sep, 'elo')

# AFC 2025
if df_afc_2025 is not None and len(df_afc_2025) > 0:
    fuentes['AFC 2025'] = (df_afc_2025, 'points')

# AFC 2026
if df_afc_2026 is not None and len(df_afc_2026) > 0:
    fuentes['AFC 2026'] = (df_afc_2026, 'points')

# CONMEBOL
if df_conmebol is not None and len(df_conmebol) > 0:
    fuentes['CONMEBOL'] = (df_conmebol, 'points')

print(f"\n📊 Mostrando Top 20 de {len(fuentes)} fuentes:\n")

# --- Recorrer cada fuente y mostrar Top 5 ---
for name, (df, score_col) in fuentes.items():
    # Ordenar por score descendente
    df_sorted = df.sort_values(score_col, ascending=False)
    
    # Tomar Top 20
    top20 = df_sorted.head(20)
    
    print(f"{'='*70}")
    print(f"📊 {name} (Top 20 por {score_col})")
    print(f"{'='*70}")
    
    # Mostrar con indice
    for i, (idx, row) in enumerate(top20.iterrows(), 1):
        team = row.get('team', row.get('club', 'Unknown'))
        score = row[score_col]
        print(f"   {i}. {team}: {score:.2f}")
    
    print()  # línea en blanco

print("\n✅ TOP 20 GLOBAL MOSTRADOS.")

TOP 20 GLOBAL DE CADA RANKING

📊 Mostrando Top 20 de 13 fuentes:

📊 G2A Lite (Top 20 por mu)
   1. PSG: 1801.51
   2. AlHilal: 1793.11
   3. Bayern: 1773.94
   4. RealMadrid: 1762.74
   5. Olympiacos: 1758.37
   6. Persepolis: 1752.38
   7. InterMilan: 1746.49
   8. Liverpool: 1742.48
   9. AlNassr: 1722.18
   10. ManCity: 1719.17
   11. Flamengo: 1714.35
   12. AlIttihad: 1711.65
   13. Arsenal: 1709.12
   14. Barcelona: 1708.54
   15. Leverkusen: 1706.26
   16. Sepahan: 1703.60
   17. AtlMadrid: 1689.76
   18. Palmeiras: 1689.04
   19. TractorSazi: 1680.10
   20. Napoli: 1678.72

📊 G2A Full (Top 20 por mu)
   1. PSG: 1807.11
   2. AlHilal: 1788.48
   3. Bayern: 1773.30
   4. Olympiacos: 1765.87
   5. RealMadrid: 1765.86
   6. Liverpool: 1749.27
   7. InterMilan: 1743.71
   8. ManCity: 1724.46
   9. Arsenal: 1715.98
   10. AlIttihad: 1715.53
   11. Barcelona: 1712.07
   12. Leverkusen: 1708.50
   13. AlNassr: 1707.20
   14. AtlMadrid: 1693.60
   15. Flamengo: 1690.83
   16. Napoli: 16

In [21]:
# %% Celda 21 - Top 5 por Liga y Media de Valoración (Comparativa Global de Modelos)

import pandas as pd
import os

# 1. Cargar dataset base para extraer mapeo de ligas
df_raw = pd.read_csv(os.path.join(BASE_DATA_DIR, "matches_final_model_clean2.csv"))

# Excluir competiciones internacionales para mapear ligas locales puras
EXCLUDED_COMPETITIONS = [
    'CONCACAFChampionsCup', 'CONCACAFChampionsLeague', 'ChampionsLeague',
    'CopaLibertadores', 'CopaSudamericana', 'EuropaConferenceLeague',
    'EuropaLeague', 'FIFAClubWorldCup', 'AFCChampionsLeagueElite', 'SuperligaArgentina'
]

df_local = df_raw[~df_raw['Competition'].isin(EXCLUDED_COMPETITIONS)].copy()

# Mapear cada equipo a la competición donde más partidos ha jugado de local
team_league_map = df_local.groupby('home_team')['Competition'].agg(
    lambda x: x.value_counts().index[0]
).to_dict()

# 2. Definir todos los modelos a comparar
modelos_a_comparar = {
    "G2A Full": df_g2a_full, 
    "G2A Lite": df_g2a_lite, 
    "ELO2025": df_elo_aug,
    "ELO2026": df_elo_sep,
    "UEFA2025": df_uefa_2025,
    "UEFA2026": df_uefa_2026,
    "AFC2025": df_afc_2025,
    "AFC2026": df_afc_2026,
    "CONMEBOL2026": df_conmebol,
    "LSTMFull": lstm_full,
    "LSTMLite": lstm_lite,
    #"ARIMAFull": arima_full,
    #"ArimaLite": arima_lite,
}

# Nombres posibles para la columna que contiene el puntaje/rating final en cada DF
posibles_cols_rating = ['mu', 'elo', 'rating', 'coef', 'points', 'score', 'pred', 'valoracion']

# 3. Procesamiento y cálculo del Top 5 por liga
for nombre_modelo, df_modelo in modelos_a_comparar.items():
    print(f"\n{'='*75}")
    print(f" TOP 5 POR LIGA Y MEDIA - {nombre_modelo.upper()}")
    print(f"{'='*75}")

    df_mod = df_modelo.copy()
    
    # Identificar automáticamente la columna de rating del modelo actual
    rating_col = next((col for col in posibles_cols_rating if col in df_mod.columns), None)
    
    if not rating_col:
        print(f"Advertencia: No se encontró una columna de valoración conocida en el modelo {nombre_modelo}.")
        print(f"Columnas disponibles: {list(df_mod.columns)}")
        continue
    
    # Asignar la liga correspondiente
    df_mod['league'] = df_mod['team'].map(team_league_map)
    df_mod = df_mod.dropna(subset=['league'])

    # Iterar sobre cada liga
    ligas_unicas = sorted(df_mod['league'].unique())

    for liga in ligas_unicas:
        # Ordenar equipos por rating descendente
        df_liga = df_mod[df_mod['league'] == liga].sort_values(by=rating_col, ascending=False)
        top5 = df_liga.head(5)
        
        if top5.empty:
            continue
            
        # Calcular media de la métrica principal
        media_top5 = top5[rating_col].mean()
        
        print(f"\n🏆 {liga} | Media del Top 5: {media_top5:.2f}")
        for i, (_, row) in enumerate(top5.iterrows(), 1):
            print(f"   {i}. {row['team']} ({rating_col}: {row[rating_col]:.2f})")


 TOP 5 POR LIGA Y MEDIA - G2A FULL

🏆 Brasileirao | Media del Top 5: 1632.10
   1. Flamengo (mu: 1690.83)
   2. Palmeiras (mu: 1672.71)
   3. Botafogo (mu: 1633.69)
   4. Mirassol (mu: 1592.01)
   5. Fluminense (mu: 1571.27)

🏆 Bundesliga | Media del Top 5: 1644.30
   1. Bayern (mu: 1773.30)
   2. Leverkusen (mu: 1708.50)
   3. Dortmund (mu: 1644.49)
   4. Leipzig (mu: 1551.51)
   5. Eintracht (mu: 1543.70)

🏆 J1League | Media del Top 5: 1598.18
   1. KashimaAntlers (mu: 1641.76)
   2. SanfrecceHiroshima (mu: 1630.10)
   3. VisselKobe (mu: 1603.27)
   4. MachidaZelvia (mu: 1561.99)
   5. KashiwaReysol (mu: 1553.76)

🏆 KLeague1 | Media del Top 5: 1546.85
   1. JeonbukHyundai (mu: 1622.82)
   2. SangjuSangmu (mu: 1543.68)
   3. DaejeonHanaCitizen (mu: 1536.70)
   4. PohangSteelers (mu: 1518.72)
   5. Gangwon (mu: 1512.33)

🏆 LaLiga | Media del Top 5: 1667.63
   1. RealMadrid (mu: 1765.86)
   2. Barcelona (mu: 1712.07)
   3. AtlMadrid (mu: 1693.60)
   4. Villarreal (mu: 1587.33)
   5. At

In [22]:
# ===== TRAZABILIDAD - FIN =====

print("="*70)
print("FIN DE EJECUCIÓN")
print("Fecha y hora UTC:", datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"))
print("Fecha y hora local:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("="*70)

FIN DE EJECUCIÓN
Fecha y hora UTC: 2026-09-03 18:59:15
Fecha y hora local: 2026-09-03 18:59:15
